# Phase 1: Model A — causal language model


## Setup


In [1]:
%pip install -q "transformers==4.53.1" "tokenizers==0.21.2" "ijson>=3.3,<4" "huggingface_hub>=0.34,<1" requests

import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["RAYON_NUM_THREADS"] = str(max(1, os.cpu_count() or 1))
os.environ["OMP_NUM_THREADS"] = str(max(1, os.cpu_count() or 1))

from pathlib import Path
import sys, json, math, random, time, gc
import numpy as np
import torch

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
os.chdir(WORK)
sys.path.insert(0, str(WORK))

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
DEVICE_IDS = list(range(min(2, GPU_COUNT)))

print("PyTorch:", torch.__version__, "primary device:", DEVICE)
print("CPU threads:", os.cpu_count())
print("Visible CUDA GPUs:", GPU_COUNT)

if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

    for i in range(GPU_COUNT):
        free, total = torch.cuda.mem_get_info(i)
        print(
            f"GPU {i}: {torch.cuda.get_device_name(i)} | "
            f"free/total GiB={free/2**30:.2f}/{total/2**30:.2f}"
        )

    if GPU_COUNT < 2:
        print("WARNING: Only one GPU is visible. Select Kaggle T4 x2 accelerator before multi-GPU training.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 956.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 44.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 68.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
PyTorch: 2.11.0+cu128 device: cuda
CPU threads: 4
GPU: Tesla T4
Free / total GiB: 14.46 / 14.56


## Configuration


In [2]:
from kaggle_secrets import UserSecretsClient

SEED = 42

TOKENIZER_PATH = Path("/kaggle/input/datasets/jaskaran7d6/tokenizer")
OUTPUT_DIR = Path("/kaggle/working/legal-lm-phase1-optimized")
TOKEN_STORE = OUTPUT_DIR / "tokens_phase1_200m_fast_v2"
RUN_NAME = "baseline_71m_200m_optimized_2gpu"

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_DATASET_REVISION = "main"

RUN_PREPROCESS = not (TOKEN_STORE / "token_manifest.json").exists()
RUN_SANITY_TESTS = True
RUN_FULL_TRAINING = False
RUN_GENERATION = True
RUN_SUBMISSION = True

AUTO_RESUME = True

USE_MULTI_GPU = True
REQUIRE_TWO_GPUS = True
SESSION_STOP_STEP = None       # e.g. 1000 for an orderly Kaggle-session boundary

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TOKEN_STORE.mkdir(parents=True, exist_ok=True)

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
assert HF_TOKEN, "Create Kaggle secret HF_TOKEN first."

print("Tokenizer:", TOKENIZER_PATH)
print("Output:", OUTPUT_DIR)
print("Token store:", TOKEN_STORE)
print("Run:", RUN_NAME)
print("RUN_PREPROCESS:", RUN_PREPROCESS)


Tokenizer: /kaggle/input/datasets/jaskaran7d6/tokenizer
Output: /kaggle/working/legal-lm-phase1-optimized
Token store: /kaggle/working/legal-lm-phase1-optimized/tokens_phase1_200m_fast_v2
Run: baseline_71m_200m_optimized
RUN_PREPROCESS: True


## Official corpus


In [3]:
from huggingface_hub import HfApi, snapshot_download

SOURCES = {
    "acts": "train_acts.json",
    "high_courts": "train_high_courts.json",
    "supreme_court": "train_supreme_court.json",
    "tribunals": "train_tribunals.json",
}

api = HfApi(token=HF_TOKEN)
info = api.dataset_info(HF_DATASET_REPO, revision=HF_DATASET_REVISION)
DATASET_REVISION = info.sha
print("Dataset commit:", DATASET_REVISION)

patterns = [
    f"Phase_1_Pretraining/Train/{filename}"
    for filename in SOURCES.values()
]

snapshot_root = Path(snapshot_download(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    revision=DATASET_REVISION,
    allow_patterns=patterns,
    token=HF_TOKEN,
    max_workers=min(4, max(1, os.cpu_count() or 1)),
))

SOURCE_PATHS = {}
for source, filename in SOURCES.items():
    path = snapshot_root / "Phase_1_Pretraining" / "Train" / filename
    assert path.is_file(), path
    SOURCE_PATHS[source] = path
    print(source, f"{path.stat().st_size/2**30:.2f} GiB", path)


Dataset commit: 89edba6a44d6bb9593fd13b6895b5cd3b06eb82e


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

Phase_1_Pretraining/Train/train_high_cou(…):   0%|          | 0.00/10.7G [00:00<?, ?B/s]

acts 0.68 GiB /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json
high_courts 9.94 GiB /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_high_courts.json
supreme_court 1.23 GiB /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json
tribunals 6.04 GiB /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json


In [4]:
from huggingface_hub import snapshot_download
from pathlib import Path
MODEL_BACKUP_REPO = "I-230482/cs781-phase1-71m-checkpoints"
TOKEN_BACKUP_REPO = "I-230482/cs781-phase1-tokenstore"
snapshot_download(
    repo_id=TOKEN_BACKUP_REPO,
    repo_type="dataset",
    token=HF_TOKEN,
    local_dir=str(TOKEN_STORE),
)

run_dir = OUTPUT_DIR / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id=MODEL_BACKUP_REPO,
    repo_type="model",
    token=HF_TOKEN,
    local_dir=str(run_dir),
)

print("Restored token store and checkpoints.")

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

acts.validation.bin:   0%|          | 0.00/2.55M [00:00<?, ?B/s]

acts.train.bin:   0%|          | 0.00/100M [00:00<?, ?B/s]

progress.json: 0.00B [00:00, ?B/s]

high_courts.train.bin:   0%|          | 0.00/100M [00:00<?, ?B/s]

high_courts.validation.bin:   0%|          | 0.00/2.50M [00:00<?, ?B/s]

... (16 outputs left out)

training.jsonl: 0.00B [00:00, ?B/s]

training_manifest.json: 0.00B [00:00, ?B/s]

training_config.json:   0%|          | 0.00/845 [00:00<?, ?B/s]

Restored token store and checkpoints.


## Preprocessing modules


In [5]:
%%writefile legal_manual_data.py
"""Fast, auditable token preprocessing for CS781 Phase 1.

Key properties:
- streams large JSON arrays with ijson (bounded RAM)
- batches documents into the Rust-backed fast tokenizer
- deterministic hash split
- writes uint16/uint32 packed token streams in large chunks
- lightweight crash-resume journal (one metadata commit per several tokenizer batches)
- memory-mapped PackedDataset for training

The hot path intentionally avoids per-document SQLite queries, per-document fsync,
and per-document binary writes.
"""
from __future__ import annotations

import bisect
import contextlib
import hashlib
import json
import math
import os
import random
import time
from pathlib import Path

import ijson
import numpy as np
import torch
from torch.utils.data import Dataset, Sampler
from transformers import AutoTokenizer


ALLOWED_REPO = "Exploration-Lab/CS-781-Capstone"
ALLOWED_FILES = {
    "acts": "train_acts.json",
    "high_courts": "train_high_courts.json",
    "supreme_court": "train_supreme_court.json",
    "tribunals": "train_tribunals.json",
}
SPLITS = ("train", "validation")
FORMAT_VERSION = "cs781-lm-tokens-fast-v2"


def require(condition, message):
    if not condition:
        raise ValueError(message)


def json_hash(obj):
    return hashlib.sha256(
        json.dumps(
            obj,
            sort_keys=True,
            ensure_ascii=False,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()


def file_hash(path, block_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            h.update(block)
    return h.hexdigest()


def write_json(path, value):
    """Atomic metadata write."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(
            value,
            f,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def tokenizer_identity(path):
    """Semantic tokenizer identity; excludes machine-specific absolute path."""
    path = Path(path).resolve()
    require(path.is_dir(), f"Missing local tokenizer: {path}")
    names = (
        "tokenizer.json",
        "tokenizer_config.json",
        "special_tokens_map.json",
    )
    hashes = {
        name: file_hash(path / name)
        for name in names
        if (path / name).is_file()
    }
    require("tokenizer.json" in hashes, "Missing tokenizer.json")
    return {
        "files": hashes,
        "sha256": json_hash(hashes),
    }


def load_tokenizer(path):
    before = tokenizer_identity(path)
    tok = AutoTokenizer.from_pretrained(
        str(Path(path).resolve()),
        local_files_only=True,
        use_fast=True,
    )
    require(tok.is_fast, "Expected custom fast tokenizer")

    special = [
        tok.pad_token_id,
        tok.unk_token_id,
        tok.bos_token_id,
        tok.eos_token_id,
    ]
    require(
        all(isinstance(i, int) and 0 <= i < len(tok) for i in special),
        "Invalid special IDs",
    )
    require(
        len(set(special)) == 4,
        "PAD, UNK, BOS and EOS must be distinct",
    )

    probe = "Section 21.  न्यायालय\n₹500 — café. Literal <pad>."
    ids = tok.encode(
        probe,
        add_special_tokens=False,
        split_special_tokens=True,
    )
    require(
        not set(ids).intersection(tok.all_special_ids),
        "Literal special-token spelling produced a special ID",
    )
    require(
        tok.decode(
            ids,
            clean_up_tokenization_spaces=False,
        ) == probe,
        "Tokenizer round-trip mismatch",
    )
    require(
        before == tokenizer_identity(path),
        "Tokenizer artifacts changed while loading",
    )

    print(
        {
            "tokenizer_vocab": len(tok),
            "special_ids": special,
            "metadata_context_limit": tok.model_max_length,
            "sha256": before["sha256"],
        }
    )
    return tok


def document_digest(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def document_split(digest, split_seed, validation_fraction):
    h = hashlib.sha256(
        f"{split_seed}:{digest}".encode("ascii")
    ).digest()
    threshold = int(validation_fraction * (1 << 64))
    return (
        "validation"
        if int.from_bytes(h[:8], "big") < threshold
        else "train"
    )


def iter_documents(source):
    """Stream one JSON-array record at a time without loading the file into RAM."""
    path = Path(source["path"])
    with path.open("rb", buffering=8 * 1024 * 1024) as raw:
        for index, record in enumerate(ijson.items(raw, "item")):
            require(
                isinstance(record, dict)
                and isinstance(record.get("text"), str),
                f"{source['source']} record {index}: expected string text",
            )
            yield index, record["text"]


def encode_document(tok, text):
    """Single-document helper used only by sanity tests."""
    ids = tok.encode(
        text,
        add_special_tokens=False,
        split_special_tokens=True,
        truncation=False,
    )
    require(
        not set(ids).intersection(tok.all_special_ids),
        "Literal text produced special IDs",
    )
    return ids + [tok.eos_token_id]


def encode_batch(tok, texts):
    """One Python call -> Rust tokenizer batch."""
    encoded = tok(
        texts,
        add_special_tokens=False,
        split_special_tokens=True,
        truncation=False,
        padding=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False,
    )["input_ids"]
    require(
        len(encoded) == len(texts),
        "Tokenizer batch length mismatch",
    )
    return encoded


def storage_dtype(vocab_size):
    require(0 < vocab_size <= 2**32, "Vocabulary cannot fit uint32")
    return np.dtype("<u2" if vocab_size <= 2**16 else "<u4")


def _new_stats():
    return {
        "raw_documents": 0,
        "raw_characters": 0,
        "empty_documents": 0,
        "whitespace_only_documents": 0,
        "train": {
            "documents": 0,
            "characters": 0,
            "tokens": 0,
            "eos_tokens": 0,
            "budget_skipped_documents": 0,
            "budget_skipped_characters": 0,
        },
        "validation": {
            "documents": 0,
            "characters": 0,
            "tokens": 0,
            "eos_tokens": 0,
            "budget_skipped_documents": 0,
            "budget_skipped_characters": 0,
        },
    }


def _quota_met(stats, limits, split):
    token_cap = limits.get(split + "_tokens")
    doc_cap = limits.get(split + "_docs")
    return (
        (token_cap is not None and stats[split]["tokens"] >= token_cap)
        or (doc_cap is not None and stats[split]["documents"] >= doc_cap)
    )


def _semantic_sources(sources):
    """Exclude absolute cache paths from the reusable fingerprint."""
    return [
        {
            "source": s["source"],
            "identity": s["identity"],
            "scope": s.get("scope", "official_training_files"),
        }
        for s in sources
    ]


def _acquire_lock(lock):
    """Break only a demonstrably stale local PID lock."""
    if lock.exists():
        stale = True
        try:
            pid = int(lock.read_text().strip())
            os.kill(pid, 0)
            stale = False
        except (ValueError, ProcessLookupError, PermissionError):
            stale = True

        if not stale:
            raise RuntimeError(
                f"Preprocessing lock is active: {lock}"
            )
        lock.unlink()

    fd = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    try:
        os.write(fd, str(os.getpid()).encode("ascii"))
    finally:
        os.close(fd)


def preprocess(sources, tok, cfg, *, interrupt_after_records=None):
    """Fast batched preprocessing with crash-safe coarse-grained resume."""
    require(
        0 < cfg["validation_fraction"] < 1,
        "Invalid validation fraction",
    )
    require(cfg["batch_docs"] > 0, "batch_docs must be positive")
    require(cfg["batch_chars"] > 0, "batch_chars must be positive")
    require(
        cfg["commit_every_batches"] > 0,
        "commit_every_batches must be positive",
    )
    require(
        set(s["source"] for s in sources) == set(ALLOWED_FILES),
        "All four official training sources are required",
    )

    for limits in cfg["source_limits"].values():
        require(
            all(
                v is None or (isinstance(v, int) and v > 0)
                for v in limits.values()
            ),
            "Source caps must be positive integers or None",
        )

    root = Path(cfg["directory"]).resolve()
    root.mkdir(parents=True, exist_ok=True)
    dtype = storage_dtype(len(tok))

    identity = {
        "format": FORMAT_VERSION,
        "tokenizer": tokenizer_identity(cfg["tokenizer_path"]),
        "vocab_size": len(tok),
        "dtype": dtype.str,
        "sources": _semantic_sources(sources),
        "split_seed": cfg["split_seed"],
        "validation_fraction": cfg["validation_fraction"],
        "source_limits": cfg["source_limits"],
        "batching": {
            "batch_docs": cfg["batch_docs"],
            "batch_chars": cfg["batch_chars"],
            "commit_every_batches": cfg["commit_every_batches"],
        },
        "duplicate_policy": (
            "retain exact duplicates; deterministic digest split guarantees "
            "an exact duplicate can never cross train/validation"
        ),
        "literal_special_tokens": "encode as ordinary byte-level text",
        "boundary": "append exactly one EOS per accepted document; no PAD",
    }
    fingerprint = json_hash(identity)

    progress_path = root / "progress.json"
    config_path = root / "preprocess_config.json"
    lock = root / "preprocess.lock"
    _acquire_lock(lock)

    try:
        if progress_path.exists():
            progress = read_json(progress_path)
            require(
                progress["fingerprint"] == fingerprint,
                "Preprocessing config/tokenizer/source changed; use a NEW directory",
            )
        else:
            progress = {
                "fingerprint": fingerprint,
                "sources": {},
            }
            write_json(progress_path, progress)

        write_json(config_path, identity)

        global_start = time.perf_counter()
        global_start_tokens = sum(
            source_state.get("stats", {}).get("train", {}).get("tokens", 0)
            + source_state.get("stats", {}).get("validation", {}).get("tokens", 0)
            for source_state in progress["sources"].values()
        )
        processed_now = 0

        for source in sources:
            name = source["source"]
            limits = cfg["source_limits"][name]

            state = progress["sources"].get(
                name,
                {
                    "cursor": -1,
                    "done": False,
                    "stats": _new_stats(),
                },
            )
            cursor = int(state["cursor"])
            stats = state["stats"]

            if state["done"]:
                print(name, "already committed; skipped")
                continue

            files = {}
            with contextlib.ExitStack() as stack:
                for split in SPLITS:
                    path = root / f"{name}.{split}.bin"
                    f = stack.enter_context(
                        path.open("r+b" if path.exists() else "w+b")
                    )
                    committed_bytes = (
                        int(stats[split]["tokens"]) * dtype.itemsize
                    )
                    require(
                        path.stat().st_size >= committed_bytes,
                        f"{path.name} is smaller than committed progress",
                    )
                    f.truncate(committed_bytes)
                    f.seek(committed_bytes)
                    files[split] = f

                last_cursor = cursor
                batches_since_commit = 0
                source_start = time.perf_counter()
                source_start_tokens = (
                    stats["train"]["tokens"]
                    + stats["validation"]["tokens"]
                )

                def committed_token_count():
                    return (
                        stats["train"]["tokens"]
                        + stats["validation"]["tokens"]
                    )

                def commit(finished=False):
                    nonlocal batches_since_commit
                    for f in files.values():
                        f.flush()
                        os.fsync(f.fileno())

                    progress["sources"][name] = {
                        "cursor": int(last_cursor),
                        "done": bool(finished),
                        "stats": stats,
                    }
                    write_json(progress_path, progress)
                    batches_since_commit = 0

                def report():
                    elapsed = max(
                        1e-9,
                        time.perf_counter() - source_start,
                    )
                    added = committed_token_count() - source_start_tokens
                    rate = added / elapsed

                    targets = [
                        limits.get("train_tokens"),
                        limits.get("validation_tokens"),
                    ]
                    target_total = (
                        sum(x for x in targets if x is not None)
                        if all(x is not None for x in targets)
                        else None
                    )
                    remaining = (
                        max(0, target_total - committed_token_count())
                        if target_total is not None
                        else None
                    )
                    eta = (
                        remaining / rate
                        if remaining is not None and rate > 0
                        else None
                    )
                    print(
                        name,
                        {
                            "train_tokens": stats["train"]["tokens"],
                            "validation_tokens": stats["validation"]["tokens"],
                            "tok_per_sec": round(rate),
                            "eta_min": (
                                round(eta / 60, 1)
                                if eta is not None
                                else None
                            ),
                            "cursor": last_cursor,
                        },
                        flush=True,
                    )

                def process_pending(pending):
                    """Tokenize pending records in one Rust-backed call."""
                    nonlocal batches_since_commit
                    if not pending:
                        return

                    texts = [item[2] for item in pending]
                    encodings = encode_batch(tok, texts)

                    chunks = {
                        "train": [],
                        "validation": [],
                    }

                    for (index, split, text), ids in zip(
                        pending,
                        encodings,
                    ):
                        if _quota_met(stats, limits, split):
                            stats[split]["budget_skipped_documents"] += 1
                            stats[split]["budget_skipped_characters"] += len(text)
                            continue

                        # split_special_tokens=True means corpus text itself must
                        # not create PAD/BOS/EOS/UNK special IDs.
                        if cfg.get("validate_special_ids", False):
                            require(
                                not set(ids).intersection(tok.all_special_ids),
                                "Corpus text produced a special ID",
                            )

                        arr = np.empty(len(ids) + 1, dtype=dtype)
                        if ids:
                            arr[:-1] = ids
                        arr[-1] = tok.eos_token_id
                        chunks[split].append(arr)

                        stats[split]["documents"] += 1
                        stats[split]["characters"] += len(text)
                        stats[split]["tokens"] += len(arr)
                        stats[split]["eos_tokens"] += 1

                    for split in SPLITS:
                        if chunks[split]:
                            packed = np.concatenate(chunks[split])
                            files[split].write(packed.tobytes(order="C"))

                    batches_since_commit += 1

                pending = []
                pending_chars = 0
                stop = False

                for index, text in iter_documents(source):
                    if index <= cursor:
                        continue

                    scan_docs = limits.get("scan_docs")
                    if scan_docs is not None and index >= scan_docs:
                        break

                    if all(
                        _quota_met(stats, limits, split)
                        for split in SPLITS
                    ):
                        stop = True
                        break

                    last_cursor = index
                    processed_now += 1

                    stats["raw_documents"] += 1
                    stats["raw_characters"] += len(text)
                    stats["empty_documents"] += int(text == "")
                    stats["whitespace_only_documents"] += int(
                        bool(text) and text.isspace()
                    )

                    digest = document_digest(text)
                    split = document_split(
                        digest,
                        cfg["split_seed"],
                        cfg["validation_fraction"],
                    )

                    # If a split is already full, avoid tokenizing documents for it.
                    if _quota_met(stats, limits, split):
                        stats[split]["budget_skipped_documents"] += 1
                        stats[split]["budget_skipped_characters"] += len(text)
                    else:
                        pending.append((index, split, text))
                        pending_chars += len(text)

                    should_flush = (
                        len(pending) >= cfg["batch_docs"]
                        or pending_chars >= cfg["batch_chars"]
                    )

                    if should_flush:
                        process_pending(pending)
                        pending = []
                        pending_chars = 0

                        if batches_since_commit >= cfg["commit_every_batches"]:
                            commit(finished=False)
                            report()

                    if (
                        interrupt_after_records is not None
                        and processed_now >= interrupt_after_records
                    ):
                        raise RuntimeError(
                            "Injected interruption for resume test"
                        )

                if pending:
                    process_pending(pending)

                complete = all(
                    _quota_met(stats, limits, split)
                    for split in SPLITS
                )
                require(
                    complete,
                    f"{name}: source ended before configured quotas were met",
                )
                commit(finished=True)
                report()

        audit = {
            name: source_state["stats"]
            for name, source_state in sorted(
                progress["sources"].items()
            )
        }

        streams = []
        for source in sources:
            name = source["source"]
            for split in SPLITS:
                path = root / f"{name}.{split}.bin"
                tokens = int(audit[name][split]["tokens"])
                expected_bytes = tokens * dtype.itemsize
                require(
                    path.stat().st_size == expected_bytes,
                    f"{path.name}: size does not match committed token count",
                )
                streams.append(
                    {
                        "source": name,
                        "split": split,
                        "file": path.name,
                        "tokens": tokens,
                        "bytes": expected_bytes,
                        "sha256": file_hash(path),
                    }
                )

        # Exact duplicates cannot cross the split because split is a pure
        # deterministic function of the exact-document SHA-256 digest.
        manifest = {
            "format": FORMAT_VERSION,
            "fingerprint": fingerprint,
            "identity": identity,
            "dtype": dtype.str,
            "vocab_size": len(tok),
            "pad_id": tok.pad_token_id,
            "eos_id": tok.eos_token_id,
            "audit": audit,
            "exact_hash_overlap": 0,
            "streams": streams,
        }
        manifest["manifest_hash"] = json_hash(manifest)
        write_json(root / "token_manifest.json", manifest)

        elapsed = time.perf_counter() - global_start
        total_tokens = sum(s["tokens"] for s in streams)
        new_tokens = max(0, total_tokens - global_start_tokens)

        print(
            {
                "preprocessing_complete": True,
                "total_tokens": total_tokens,
                "new_tokens_this_session": new_tokens,
                "elapsed_min": round(elapsed / 60, 2),
                "session_tok_per_sec": (
                    round(new_tokens / elapsed)
                    if elapsed > 0
                    else None
                ),
                "exact_hash_overlap": 0,
            },
            flush=True,
        )
        return manifest
    finally:
        lock.unlink(missing_ok=True)


def packing_report(manifest, context_length):
    rows = []
    for stream in manifest["streams"]:
        blocks, remainder = divmod(
            stream["tokens"],
            context_length,
        )
        rows.append(
            {
                "source": stream["source"],
                "split": stream["split"],
                "produced_tokens": stream["tokens"],
                "complete_blocks": blocks,
                "used_input_tokens": blocks * context_length,
                "retained_but_unused_tail_tokens": remainder,
            }
        )
    return rows


class PackedDataset(Dataset):
    """Memory-map packed token streams. Training returns input_ids only."""

    def __init__(
        self,
        directory,
        split,
        context_length,
        verify_hashes=True,
    ):
        self.root = Path(directory).resolve()
        self.manifest = read_json(
            self.root / "token_manifest.json"
        )

        check = dict(self.manifest)
        checksum = check.pop("manifest_hash")
        require(
            json_hash(check) == checksum,
            "Token manifest checksum mismatch",
        )

        require(
            split in SPLITS and context_length >= 2,
            "Invalid packed-dataset configuration",
        )

        self.context = int(context_length)
        self.dtype = np.dtype(self.manifest["dtype"])
        self.streams = []
        self.ends = []
        self.maps = {}

        total = 0
        for stream in self.manifest["streams"]:
            if stream["split"] != split:
                continue

            path = self.root / stream["file"]
            require(
                path.stat().st_size == stream["bytes"],
                "Token storage size changed",
            )
            if verify_hashes:
                require(
                    file_hash(path) == stream["sha256"],
                    f"Token storage hash changed: {path.name}",
                )

            blocks = stream["tokens"] // self.context
            if blocks:
                self.streams.append(stream)
                total += blocks
                self.ends.append(total)

        require(
            total > 0,
            f"No complete {split} blocks",
        )

    def __len__(self):
        return self.ends[-1]

    def __getitem__(self, index):
        if index < 0:
            index += len(self)
        require(0 <= index < len(self), "Block index out of range")

        file_index = bisect.bisect_right(
            self.ends,
            index,
        )
        prior = (
            self.ends[file_index - 1]
            if file_index
            else 0
        )

        if file_index not in self.maps:
            self.maps[file_index] = np.memmap(
                self.root / self.streams[file_index]["file"],
                dtype=self.dtype,
                mode="r",
            )

        start = (index - prior) * self.context
        view = self.maps[file_index][
            start : start + self.context
        ]

        # Embedding indices must be int64. Hash verification at dataset open makes
        # repeated per-item min/max/PAD scans unnecessary.
        ids = torch.from_numpy(
            np.asarray(view, dtype=np.int64).copy()
        )
        return {"input_ids": ids}

    def __getstate__(self):
        state = self.__dict__.copy()
        state["maps"] = {}
        return state

    def close(self):
        for mapping in self.maps.values():
            mapping._mmap.close()
        self.maps.clear()

    def __del__(self):
        try:
            self.close()
        except Exception:
            pass


class ResumableBatches(Sampler):
    """O(1)-memory deterministic affine permutation of block indices."""

    def __init__(
        self,
        size,
        batch_size,
        start_microbatch,
        microbatches,
        seed,
    ):
        self.size = int(size)
        self.batch_size = int(batch_size)
        self.start = int(start_microbatch)
        self.count = int(microbatches)
        self.seed = int(seed)
        require(
            self.size > 0
            and self.batch_size > 0
            and self.count >= 0,
            "Invalid sampler dimensions",
        )

    def __len__(self):
        return self.count

    def __iter__(self):
        epoch_cache = None
        a, b = 1, 0

        for micro in range(
            self.start,
            self.start + self.count,
        ):
            indices = []

            for absolute in range(
                micro * self.batch_size,
                (micro + 1) * self.batch_size,
            ):
                epoch, offset = divmod(
                    absolute,
                    self.size,
                )

                if epoch != epoch_cache:
                    rng = random.Random(
                        f"{self.seed}:block-order:{epoch}"
                    )
                    a = rng.randrange(
                        1,
                        self.size + 1,
                    )
                    while math.gcd(a, self.size) != 1:
                        a = a % self.size + 1
                    b = rng.randrange(self.size)
                    epoch_cache = epoch

                indices.append(
                    (a * offset + b) % self.size
                )

            yield indices


class EvenlySpacedSubset(Dataset):
    def __init__(self, dataset, count):
        self.dataset = dataset
        self.count = int(count)

    def __len__(self):
        return self.count

    def __getitem__(self, index):
        return self.dataset[
            index * len(self.dataset) // self.count
        ]


def seed_worker(worker_id):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed)
    random.seed(seed)


def loader_options(cfg):
    workers = int(cfg["num_workers"])
    require(workers >= 0, "num_workers must be nonnegative")

    out = {
        "num_workers": workers,
        "pin_memory": bool(cfg["pin_memory"]),
        "worker_init_fn": seed_worker,
        "generator": torch.Generator().manual_seed(
            int(cfg["seed"])
        ),
    }

    if workers:
        out.update(
            persistent_workers=bool(
                cfg["persistent_workers"]
            ),
            prefetch_factor=int(
                cfg["prefetch_factor"]
            ),
        )
    return out


Writing legal_manual_data.py


In [6]:
import legal_manual_data as data

tokenizer = data.load_tokenizer(TOKENIZER_PATH)
VOCAB_SIZE = len(tokenizer)
TOKENIZER_HASH = data.tokenizer_identity(TOKENIZER_PATH)["sha256"]

print("Vocabulary:", VOCAB_SIZE)
print(
    {
        "pad": tokenizer.pad_token_id,
        "unk": tokenizer.unk_token_id,
        "bos": tokenizer.bos_token_id,
        "eos": tokenizer.eos_token_id,
    }
)
assert VOCAB_SIZE == 32_000
assert [
    tokenizer.pad_token_id,
    tokenizer.unk_token_id,
    tokenizer.bos_token_id,
    tokenizer.eos_token_id,
] == [0, 1, 2, 3]


{'tokenizer_vocab': 32000, 'special_ids': [0, 1, 2, 3], 'metadata_context_limit': 2048, 'sha256': '3163063597ba6358484fc37200c76f548040a2a58186862b519904ba427deb2b'}
Vocabulary: 32000
{'pad': 0, 'unk': 1, 'bos': 2, 'eos': 3}


## Tokenizer sanity check


In [7]:
probe = "Section 21.  W.P.(C) No. 1234/2021: न्यायालय — ₹500.\nLiteral <pad>."

single = data.encode_document(tokenizer, probe)
batch = data.encode_batch(tokenizer, [probe])[0]

assert single[:-1] == batch
assert single[-1] == tokenizer.eos_token_id
assert not set(batch).intersection(tokenizer.all_special_ids)
assert tokenizer.decode(
    batch,
    clean_up_tokenization_spaces=False,
) == probe

print("Tokenizer sanity check passed.")


Tokenizer sanity check passed.


## Corpus preprocessing


In [8]:
PREPROCESS = dict(
    directory=TOKEN_STORE,
    tokenizer_path=TOKENIZER_PATH,
    split_seed=42,
    validation_fraction=0.05,

    # Fast-tokenizer batching
    batch_docs=64,
    batch_chars=2_000_000,
    commit_every_batches=20,

    # The tokenizer itself guarantees IDs are in range. The explicit literal
    # special-token behavior was already checked above, so skip a Python set()
    # scan over every encoded document in the production hot path.
    validate_special_ids=False,

    source_limits={
        name: dict(
            train_tokens=50_000_000,
            validation_tokens=1_250_000,
            train_docs=None,
            validation_docs=None,
            scan_docs=None,
        )
        for name in data.ALLOWED_FILES
    },
)

sources = []
for source, path in SOURCE_PATHS.items():
    filename = f"Phase_1_Pretraining/Train/{SOURCES[source]}"
    sources.append({
        "source": source,
        "path": str(path),
        "identity": {
            "repo": HF_DATASET_REPO,
            "commit": DATASET_REVISION,
            "file": filename,
            "bytes": path.stat().st_size,
        },
        "scope": "official_training_files",
    })

print("Target train tokens:", 4 * 50_000_000)
print("Target validation tokens:", 4 * 1_250_000)
print("Batch docs:", PREPROCESS["batch_docs"])
print("Batch chars:", PREPROCESS["batch_chars"])


Target train tokens: 200000000
Target validation tokens: 5000000
Batch docs: 64
Batch chars: 2000000


In [9]:
if RUN_PREPROCESS:
    manifest = data.preprocess(
        sources,
        tokenizer,
        PREPROCESS,
    )
else:
    print("Preprocessing skipped; using existing token store.")
    manifest = data.read_json(TOKEN_STORE / "token_manifest.json")

assert manifest["exact_hash_overlap"] == 0

for source, stats in manifest["audit"].items():
    print(
        source,
        {
            split: {
                "documents": stats[split]["documents"],
                "tokens": stats[split]["tokens"],
            }
            for split in ("train", "validation")
        },
    )

print("Manifest ready:", TOKEN_STORE / "token_manifest.json")


acts already committed; skipped
high_courts already committed; skipped
supreme_court already committed; skipped
tribunals already committed; skipped
{'preprocessing_complete': True, 'total_tokens': 205136117, 'new_tokens_this_session': 0, 'elapsed_min': 0.02, 'session_tok_per_sec': 0, 'exact_hash_overlap': 0}
acts {'train': {'documents': 5963, 'tokens': 50091621}, 'validation': {'documents': 149, 'tokens': 1273494}}
high_courts {'train': {'documents': 89847, 'tokens': 50000807}, 'validation': {'documents': 2215, 'tokens': 1250583}}
supreme_court {'train': {'documents': 10317, 'tokens': 50001382}, 'validation': {'documents': 169, 'tokens': 1259422}}
tribunals {'train': {'documents': 15891, 'tokens': 50003901}, 'validation': {'documents': 146, 'tokens': 1254907}}
Manifest ready: /kaggle/working/legal-lm-phase1-optimized/tokens_phase1_200m_fast_v2/token_manifest.json


## Packed-data report


In [10]:
for row in data.packing_report(manifest, 2048):
    print(row)


{'source': 'acts', 'split': 'train', 'produced_tokens': 50091621, 'complete_blocks': 24458, 'used_input_tokens': 50089984, 'retained_but_unused_tail_tokens': 1637}
{'source': 'acts', 'split': 'validation', 'produced_tokens': 1273494, 'complete_blocks': 621, 'used_input_tokens': 1271808, 'retained_but_unused_tail_tokens': 1686}
{'source': 'high_courts', 'split': 'train', 'produced_tokens': 50000807, 'complete_blocks': 24414, 'used_input_tokens': 49999872, 'retained_but_unused_tail_tokens': 935}
{'source': 'high_courts', 'split': 'validation', 'produced_tokens': 1250583, 'complete_blocks': 610, 'used_input_tokens': 1249280, 'retained_but_unused_tail_tokens': 1303}
{'source': 'supreme_court', 'split': 'train', 'produced_tokens': 50001382, 'complete_blocks': 24414, 'used_input_tokens': 49999872, 'retained_but_unused_tail_tokens': 1510}
{'source': 'supreme_court', 'split': 'validation', 'produced_tokens': 1259422, 'complete_blocks': 614, 'used_input_tokens': 1257472, 'retained_but_unused_ta

## Model A architecture


In [11]:
MODEL_CONFIG = dict(
    vocab_size=32_000,
    context_length=2048,

    d_model=640,
    n_layers=10,
    n_heads=10,
    d_ff=2560,

    dropout=0.1,
    position="learned",
    normalization="layernorm",
    norm_placement="pre",
    norm_epsilon=1e-5,
    activation="gelu",

    tie_embeddings=True,

    # Optimized SDPA substantially reduces attention memory. Leave this False
    # unless even microbatch=1 fails the fit probe.
    activation_checkpointing=False,
)

assert MODEL_CONFIG["d_model"] % MODEL_CONFIG["n_heads"] == 0
print("Head dimension:", MODEL_CONFIG["d_model"] // MODEL_CONFIG["n_heads"])


Head dimension: 64


In [12]:
%%writefile legal_manual_model.py
"""Optimized manual decoder for CS781 Phase 1.

The architecture remains a manually implemented dense causal Transformer baseline.
Optimizations are kernel/implementation changes: fused QKV projection, PyTorch SDPA,
fused layer_norm primitive, optional activation checkpointing, and KV-cached generation.
No pretrained language-model weights are used.
"""
import contextlib
import math
import re
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint


class TokenEmbedding(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.table = nn.Embedding(cfg["vocab_size"], cfg["d_model"])

    def forward(self, ids):
        return self.table(ids)


class LearnedPositions(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg["position"] == "learned"
        self.table = nn.Embedding(cfg["context_length"], cfg["d_model"])
        self.register_buffer(
            "position_ids",
            torch.arange(cfg["context_length"], dtype=torch.long),
            persistent=False,
        )

    def forward(self, x, offset=0):
        end = offset + x.shape[1]
        assert end <= self.table.num_embeddings
        pos = self.position_ids[offset:end]
        return x + self.table(pos)[None, :, :]


class CausalAttention(nn.Module):
    """Manual MHA projections + optimized scaled-dot-product attention kernel."""

    def __init__(self, cfg):
        super().__init__()
        d = cfg["d_model"]
        self.heads = cfg["n_heads"]
        assert d % self.heads == 0
        self.head_dim = d // self.heads

        # One projection instead of three separate GEMMs; parameter count is unchanged.
        self.qkv = nn.Linear(d, 3 * d)
        self.out = nn.Linear(d, d)
        self.attn_dropout = float(cfg["dropout"])

    def _project(self, x):
        B, T, D = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        return (
            q.transpose(1, 2),  # [B,H,T,h]
            k.transpose(1, 2),
            v.transpose(1, 2),
        )

    def forward(self, x, past_kv=None, use_cache=False):
        q, k, v = self._project(x)

        if past_kv is None:
            # Training and prompt prefill: ordinary causal self-attention.
            is_causal = True
        else:
            # Cached generation only feeds one new token at a time.
            assert q.shape[-2] == 1, "Cached continuation expects one token per step."
            past_k, past_v = past_kv
            k = torch.cat((past_k, k), dim=-2)
            v = torch.cat((past_v, v), dim=-2)
            # All keys are past/current; there are no future keys to mask.
            is_causal = False

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.attn_dropout if self.training else 0.0,
            is_causal=is_causal,
        )
        B, H, T, h = y.shape
        y = y.transpose(1, 2).contiguous().view(B, T, H * h)
        y = self.out(y)

        if use_cache:
            return y, (k, v)
        return y


class Normalization(nn.Module):
    """Learned LayerNorm parameters with PyTorch's fused layer_norm primitive."""

    def __init__(self, cfg):
        super().__init__()
        assert cfg["normalization"] == "layernorm"
        d = cfg["d_model"]
        self.weight = nn.Parameter(torch.ones(d))
        self.bias = nn.Parameter(torch.zeros(d))
        self.eps = cfg["norm_epsilon"]

    def forward(self, x):
        return F.layer_norm(
            x,
            (x.shape[-1],),
            self.weight,
            self.bias,
            self.eps,
        )


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg["activation"] == "gelu"
        self.up = nn.Linear(cfg["d_model"], cfg["d_ff"])
        self.down = nn.Linear(cfg["d_ff"], cfg["d_model"])
        self.dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        return self.down(self.dropout(F.gelu(self.up(x))))


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = Normalization(cfg)
        self.norm2 = Normalization(cfg)
        self.attention = CausalAttention(cfg)
        self.mlp = FeedForward(cfg)
        self.dropout = nn.Dropout(cfg["dropout"])
        self.placement = cfg["norm_placement"]
        assert self.placement in ("pre", "post")

    def forward(self, x, past_kv=None, use_cache=False):
        if self.placement == "pre":
            h = self.norm1(x)
            if use_cache:
                attn, new_kv = self.attention(h, past_kv=past_kv, use_cache=True)
            else:
                attn = self.attention(h)
            x = x + self.dropout(attn)
            x = x + self.dropout(self.mlp(self.norm2(x)))
        else:
            if use_cache:
                attn, new_kv = self.attention(x, past_kv=past_kv, use_cache=True)
            else:
                attn = self.attention(x)
            x = self.norm1(x + self.dropout(attn))
            x = self.norm2(x + self.dropout(self.mlp(x)))

        if use_cache:
            return x, new_kv
        return x


def initialize_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, Normalization):
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)


class LegalLanguageModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = dict(cfg)
        assert cfg["d_model"] % cfg["n_heads"] == 0

        self.tokens = TokenEmbedding(cfg)
        self.positions = LearnedPositions(cfg)
        self.dropout = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList(
            [DecoderBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = Normalization(cfg)
        self.lm_head = nn.Linear(
            cfg["d_model"],
            cfg["vocab_size"],
            bias=False,
        )

        self.apply(initialize_weights)
        if cfg["tie_embeddings"]:
            self.lm_head.weight = self.tokens.table.weight

    def forward(self, ids, past_key_values=None, use_cache=False):
        assert ids.ndim == 2 and 0 < ids.shape[1] <= self.cfg["context_length"]

        if past_key_values is None:
            offset = 0
            if use_cache:
                past_key_values = [None] * len(self.blocks)
        else:
            assert use_cache
            assert len(past_key_values) == len(self.blocks)
            offset = past_key_values[0][0].shape[-2]

        assert offset + ids.shape[1] <= self.cfg["context_length"]

        x = self.tokens(ids)
        x = self.positions(x, offset=offset)
        x = self.dropout(x)

        if use_cache:
            new_cache = []
            for i, block in enumerate(self.blocks):
                x, kv = block(
                    x,
                    past_kv=past_key_values[i],
                    use_cache=True,
                )
                new_cache.append(kv)
            logits = self.lm_head(self.final_norm(x))
            return logits, new_cache

        for block in self.blocks:
            if self.training and self.cfg["activation_checkpointing"]:
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)

        return self.lm_head(self.final_norm(x))


def parameter_counts(model):
    seen = set()
    total = 0
    for p in model.parameters():
        if id(p) not in seen:
            total += p.numel()
            seen.add(id(p))
    return {
        "total": total,
        "trainable": sum(p.numel() for p in model.parameters() if p.requires_grad),
    }


def causal_loss(logits, ids):
    """logits[:,i] predicts ids[:,i+1]. Autocast handles CE precision safely."""
    return F.cross_entropy(
        logits[:, :-1].reshape(-1, logits.shape[-1]),
        ids[:, 1:].reshape(-1),
    )


def _inference_precision(device, requested="auto"):
    if device.type != "cuda":
        return "fp32"
    if requested == "auto":
        return "bf16" if torch.cuda.is_bf16_supported() else "fp16"
    return requested


def _amp_context(device, precision):
    if precision == "fp32":
        return contextlib.nullcontext()
    dtype = torch.bfloat16 if precision == "bf16" else torch.float16
    return torch.autocast(device_type=device.type, dtype=dtype)


@torch.inference_mode()
def generate_tokens(
    model,
    tokenizer,
    prompt,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.95,
    seed=42,
    precision="auto",
    stop_after_words=None,
):
    """KV-cached autoregressive generation.

    The full prompt is evaluated once. Every later step evaluates only the new token.
    """
    assert max_new_tokens > 0
    assert temperature >= 0
    assert 0 < top_p <= 1

    ids = tokenizer.encode(
        prompt,
        add_special_tokens=False,
        split_special_tokens=True,
    )
    assert ids, "Supply a nonempty prompt."
    assert len(ids) + max_new_tokens <= model.cfg["context_length"], (
        "Prompt + max_new_tokens exceeds model context."
    )

    device = next(model.parameters()).device
    precision = _inference_precision(device, precision)
    rng = torch.Generator(device=device).manual_seed(seed)

    was_training = model.training
    model.eval()
    generated = []

    try:
        x = torch.tensor([ids], dtype=torch.long, device=device)
        with _amp_context(device, precision):
            logits, cache = model(x, use_cache=True)

        for step in range(max_new_tokens):
            next_logits = logits[0, -1].float()

            for special in (
                tokenizer.pad_token_id,
                tokenizer.bos_token_id,
                tokenizer.unk_token_id,
            ):
                if special is not None:
                    next_logits[special] = float("-inf")

            if temperature == 0:
                next_id = int(next_logits.argmax())
            else:
                sorted_logits, order = torch.sort(
                    next_logits / temperature,
                    descending=True,
                )
                probs = sorted_logits.softmax(-1)
                remove = probs.cumsum(-1) - probs > top_p
                sorted_logits[remove] = float("-inf")
                next_id = int(
                    order[
                        torch.multinomial(
                            sorted_logits.softmax(-1),
                            1,
                            generator=rng,
                        )
                    ]
                )

            if next_id == tokenizer.eos_token_id:
                break

            generated.append(next_id)

            # Check words every 8 generated tokens; avoids generating hundreds of
            # unnecessary extra tokens once the required 128 words are available.
            if (
                stop_after_words is not None
                and (len(generated) % 8 == 0 or step + 1 == max_new_tokens)
            ):
                partial = tokenizer.decode(
                    generated,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=False,
                )
                if len(re.findall(r"\S+", partial)) >= stop_after_words:
                    break

            x = torch.tensor([[next_id]], dtype=torch.long, device=device)
            with _amp_context(device, precision):
                logits, cache = model(
                    x,
                    past_key_values=cache,
                    use_cache=True,
                )

        return tokenizer.decode(
            generated,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
    finally:
        model.train(was_training)


def continuation_words(
    model,
    tokenizer,
    prompt,
    words=128,
    max_new_tokens=512,
    **kwargs,
):
    text = generate_tokens(
        model,
        tokenizer,
        prompt,
        max_new_tokens=max_new_tokens,
        stop_after_words=words,
        **kwargs,
    )
    spans = list(re.finditer(r"\S+", text))
    if len(spans) < words:
        raise ValueError(
            f"Only {len(spans)} words generated; need {words}. "
            "Review EOS/token/context limits."
        )
    return text[: spans[words - 1].end()].strip()


Writing legal_manual_model.py


In [13]:
from legal_manual_model import (
    LegalLanguageModel,
    causal_loss,
    continuation_words,
    parameter_counts,
)

with torch.device("meta"):
    summary_model = LegalLanguageModel(MODEL_CONFIG)

counts = parameter_counts(summary_model)
print(counts)
print("Parameters (M):", round(counts["total"] / 1e6, 2))
assert 50_000_000 <= counts["total"] <= 300_000_000
assert summary_model.lm_head.weight is summary_model.tokens.table.weight
del summary_model


{'total': 71027200, 'trainable': 71027200}
Parameters (M): 71.03


## Sanity tests for the optimized attention/cache path


In [14]:
if RUN_SANITY_TESTS:
    tiny_cfg = dict(
        MODEL_CONFIG,
        vocab_size=257,
        context_length=32,
        d_model=32,
        n_layers=2,
        n_heads=4,
        d_ff=64,
        dropout=0.0,
        activation_checkpointing=False,
    )

    tiny = LegalLanguageModel(tiny_cfg).to(DEVICE).eval()

    a = torch.randint(0, 257, (2, 12), device=DEVICE)
    b = a.clone()
    b[:, 9] = (b[:, 9] + 1) % 257

    with torch.no_grad():
        out_a = tiny(a)
        out_b = tiny(b)

    # Changing a future token cannot affect earlier logits.
    torch.testing.assert_close(
        out_a[:, :9],
        out_b[:, :9],
        rtol=1e-4,
        atol=1e-5,
    )

    # Cached last-token logits should match the full-prefix last-token logits.
    x = torch.randint(0, 257, (1, 10), device=DEVICE)
    with torch.no_grad():
        full = tiny(x)
        _, cache = tiny(x[:, :-1], use_cache=True)
        cached, _ = tiny(
            x[:, -1:],
            past_key_values=cache,
            use_cache=True,
        )

    torch.testing.assert_close(
        full[:, -1],
        cached[:, -1],
        rtol=1e-4,
        atol=1e-5,
    )

    print("Optimized model sanity tests passed.")
    del tiny, a, b, out_a, out_b, x, full, cached, cache
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
else:
    print("Sanity tests disabled.")


Optimized model sanity tests passed.


## Optimized training configuration


In [15]:
TRAIN_CONFIG = dict(
    # Autotuner preserves this effective token count.
    tokens_per_optimizer_step=65_536,
    auto_microbatch=True,
    microbatch_candidates=[8, 4, 2],
    optimizer_memory_reserve_gib=2.5,

    # One-process/two-GPU DataParallel. Physical batch size is GLOBAL.
    use_multi_gpu=USE_MULTI_GPU,
    require_two_gpus=REQUIRE_TWO_GPUS,
    device_ids=DEVICE_IDS,

    # Initial values are replaced by the fit probe on a fresh run.
    physical_batch_size=2,
    gradient_accumulation_steps=16,
    eval_batch_size=4,

    learning_rate=2e-4,
    weight_decay=0.1,
    warmup_ratio=0.02,
    max_grad_norm=1.0,

    train_token_budget=200_000_000,
    validation_token_budget=5_000_000,
    eval_token_budget=1_000_000,

    logging_steps=25,
    eval_steps=500,
    checkpoint_steps=250,
    keep_checkpoints=2,

    optimizer="adamw",
    fused_optimizer=True,
    scheduler="cosine",
    betas=[0.9, 0.95],
    epsilon=1e-8,

    precision="auto",
    strict_determinism=False,
    seed=SEED,

    # Memmap block reads are cheap but prefetching keeps CPU work off the GPU path.
    num_workers=2,
    pin_memory=DEVICE.type == "cuda",
    persistent_workers=True,
    prefetch_factor=4,
)

print("Train token budget:", TRAIN_CONFIG["train_token_budget"])
print("Validation token budget:", TRAIN_CONFIG["validation_token_budget"])


Train token budget: 200000000
Validation token budget: 5000000


## Training and checkpoints


In [16]:
%%writefile legal_manual_training.py
"""Optimized training/checkpoint loop for the manual CS781 model."""
from __future__ import annotations

import contextlib
import json
import math
import os
import platform
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

import legal_manual_data as data
from legal_manual_model import (
    LegalLanguageModel,
    causal_loss,
    parameter_counts,
)


class ReplicaAutocast(torch.nn.Module):
    """Enter autocast inside each DataParallel replica thread.

    torch.autocast state is thread-local, so an outer autocast context alone
    is not sufficient for nn.DataParallel replica threads.
    """

    def __init__(self, module, precision):
        super().__init__()
        self.module = module
        self.precision = precision

    @property
    def cfg(self):
        return self.module.cfg

    def forward(self, *args, **kwargs):
        if self.precision == "fp32":
            return self.module(*args, **kwargs)

        dtype = (
            torch.bfloat16
            if self.precision == "bf16"
            else torch.float16
        )
        with torch.autocast(
            device_type="cuda",
            dtype=dtype,
        ):
            return self.module(*args, **kwargs)


class ConfigDataParallel(torch.nn.DataParallel):
    @property
    def cfg(self):
        return self.module.cfg


def unwrap_model(model):
    """Return the underlying LegalLanguageModel for save/load/identity."""
    while isinstance(
        model,
        (torch.nn.DataParallel, ReplicaAutocast),
    ):
        model = model.module
    return model


def prepare_training_model(model, tc, device):
    """Wrap a CUDA model so one notebook process uses both visible T4s."""
    precision = precision_for(device, tc["precision"])

    if not tc.get("use_multi_gpu", False):
        return model

    device_ids = [
        int(i)
        for i in tc.get("device_ids", [])
        if int(i) < torch.cuda.device_count()
    ]

    if tc.get("require_two_gpus", False) and len(device_ids) < 2:
        raise RuntimeError(
            "Two-GPU training requested but fewer than two configured CUDA devices "
            "are visible. In Kaggle select the T4 x2 accelerator and restart."
        )

    if len(device_ids) < 2:
        print("Multi-GPU requested but only one configured GPU is visible; using one GPU.")
        return model

    if device != torch.device(f"cuda:{device_ids[0]}"):
        raise ValueError(
            f"Primary device must be cuda:{device_ids[0]} for DataParallel; got {device}."
        )

    # Autocast must be entered inside every replica's worker thread.
    wrapped = ReplicaAutocast(model, precision)

    parallel = ConfigDataParallel(
        wrapped,
        device_ids=device_ids,
        output_device=device_ids[0],
    )

    print(
        "DataParallel enabled:",
        {
            "device_ids": device_ids,
            "precision": precision,
            "global_physical_batch": tc["physical_batch_size"],
        },
    )
    return parallel


def seed_all(seed, strict_determinism=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Strict deterministic kernels are useful for debugging, but noticeably slower.
    torch.use_deterministic_algorithms(
        bool(strict_determinism),
        warn_only=not bool(strict_determinism),
    )

    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = not bool(strict_determinism)
        # Harmless on T4; accelerates FP32 matmuls on Ampere+ GPUs.
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")


def runtime():
    return {
        "torch": str(torch.__version__),
        "numpy": np.__version__,
        "python": platform.python_version(),
        "cuda": torch.version.cuda,
        "device": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else "cpu"
        ),
        "visible_gpu_count": (
            torch.cuda.device_count()
            if torch.cuda.is_available()
            else 0
        ),
        "visible_gpus": (
            [
                torch.cuda.get_device_name(i)
                for i in range(torch.cuda.device_count())
            ]
            if torch.cuda.is_available()
            else []
        ),
    }


def precision_for(device, requested="auto"):
    if requested == "auto":
        if device.type != "cuda":
            return "fp32"
        return (
            "bf16"
            if torch.cuda.is_bf16_supported()
            else "fp16"
        )

    assert requested in ("fp32", "fp16", "bf16")
    assert requested == "fp32" or device.type == "cuda"
    assert (
        requested != "bf16"
        or torch.cuda.is_bf16_supported()
    )
    return requested


def amp(device, precision):
    if precision == "fp32":
        return contextlib.nullcontext()

    dtype = (
        torch.bfloat16
        if precision == "bf16"
        else torch.float16
    )
    return torch.autocast(
        device_type=device.type,
        dtype=dtype,
    )


def plan(mc, tc):
    required_ints = (
        "physical_batch_size",
        "gradient_accumulation_steps",
        "train_token_budget",
        "validation_token_budget",
        "eval_token_budget",
        "logging_steps",
        "eval_steps",
        "checkpoint_steps",
        "keep_checkpoints",
    )
    for key in required_ints:
        assert isinstance(tc[key], int) and tc[key] > 0, key

    assert 0 <= tc["warmup_ratio"] < 1
    assert tc["learning_rate"] > 0
    assert tc["max_grad_norm"] > 0

    micro_tokens = (
        tc["physical_batch_size"]
        * mc["context_length"]
    )
    effective_tokens = (
        micro_tokens
        * tc["gradient_accumulation_steps"]
    )
    steps = (
        tc["train_token_budget"]
        // effective_tokens
    )
    assert steps > 0

    return {
        "tokens_per_microbatch": micro_tokens,
        "effective_batch_size": (
            tc["physical_batch_size"]
            * tc["gradient_accumulation_steps"]
        ),
        "tokens_per_step": effective_tokens,
        "steps": steps,
        "planned_train_tokens": steps * effective_tokens,
        "warmup": math.ceil(
            steps * tc["warmup_ratio"]
        ),
    }


def optimizer_scheduler(model, mc, tc):
    assert tc["optimizer"] == "adamw"

    decay = []
    no_decay = []

    for p in model.parameters():
        (
            decay
            if p.ndim >= 2
            else no_decay
        ).append(p)

    groups = [
        {
            "params": decay,
            "weight_decay": tc["weight_decay"],
        },
        {
            "params": no_decay,
            "weight_decay": 0.0,
        },
    ]

    kwargs = dict(
        lr=tc["learning_rate"],
        betas=tuple(tc["betas"]),
        eps=tc["epsilon"],
    )

    # Fused AdamW reduces many small CUDA kernel launches.
    use_fused = (
        tc.get("fused_optimizer", True)
        and next(model.parameters()).is_cuda
    )
    if use_fused:
        kwargs["fused"] = True

    try:
        optimizer = torch.optim.AdamW(
            groups,
            **kwargs,
        )
    except (TypeError, RuntimeError):
        kwargs.pop("fused", None)
        optimizer = torch.optim.AdamW(
            groups,
            **kwargs,
        )
        use_fused = False

    p = plan(mc, tc)

    def schedule(step):
        if p["warmup"] and step < p["warmup"]:
            return (step + 1) / p["warmup"]

        progress = min(
            1.0,
            (step - p["warmup"])
            / max(1, p["steps"] - p["warmup"]),
        )

        if tc["scheduler"] == "cosine":
            return 0.5 * (
                1 + math.cos(math.pi * progress)
            )
        if tc["scheduler"] == "linear":
            return 1 - progress

        assert tc["scheduler"] == "constant"
        return 1.0

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        schedule,
    )

    return optimizer, scheduler, use_fused


def loader_options(tc):
    return data.loader_options(
        {
            "num_workers": tc["num_workers"],
            "pin_memory": tc["pin_memory"],
            "persistent_workers": tc[
                "persistent_workers"
            ],
            "prefetch_factor": tc[
                "prefetch_factor"
            ],
            "seed": tc["seed"],
        }
    )


@torch.no_grad()
def validate(
    model,
    dataset,
    token_budget,
    batch_size,
    tc,
    device,
    precision,
):
    block_budget = max(
        1,
        token_budget // model.cfg["context_length"],
    )
    count = min(
        len(dataset),
        block_budget,
    )

    subset = data.EvenlySpacedSubset(
        dataset,
        count,
    )
    loader = DataLoader(
        subset,
        batch_size=batch_size,
        **loader_options(tc),
    )

    previous = model.training
    model.eval()

    # Keep the reduction on GPU; synchronize only once after validation.
    total_loss = torch.zeros(
        (),
        device=device,
        dtype=torch.float64,
    )
    targets = 0

    try:
        for batch in loader:
            ids = batch["input_ids"].to(
                device,
                non_blocking=True,
            )
            with amp(device, precision):
                loss = causal_loss(
                    model(ids),
                    ids,
                )

            n = ids.shape[0] * (
                ids.shape[1] - 1
            )
            total_loss += loss.detach().double() * n
            targets += n
    finally:
        model.train(previous)

    loss_value = float(
        (total_loss / targets).item()
    )

    return {
        "validation_loss": loss_value,
        "validation_perplexity": (
            math.exp(loss_value)
            if loss_value < 700
            else None
        ),
        "validation_targets": targets,
    }


def autotune_microbatch(model, tc, device):
    """Choose the largest safe GLOBAL physical batch.

    With DataParallel and two GPUs, candidate batch sizes must be >=2 and
    divisible by the number of participating GPUs so both T4s receive work.
    The effective token count per optimizer step remains unchanged.
    """
    if (
        device.type != "cuda"
        or not tc.get("auto_microbatch", True)
    ):
        return dict(tc)

    target_tokens = int(tc["tokens_per_optimizer_step"])
    context = model.cfg["context_length"]
    precision = precision_for(device, tc["precision"])

    use_multi = bool(tc.get("use_multi_gpu", False))
    device_ids = [
        int(i)
        for i in tc.get("device_ids", [])
        if int(i) < torch.cuda.device_count()
    ]
    world = len(device_ids) if use_multi else 1

    if tc.get("require_two_gpus", False) and world < 2:
        raise RuntimeError(
            "Two-GPU training requested but two configured CUDA devices are not visible."
        )

    original_training = model.training
    model.train()
    chosen = None

    for batch_size in tc["microbatch_candidates"]:
        batch_size = int(batch_size)

        if use_multi and world >= 2:
            if batch_size < world or batch_size % world != 0:
                continue

        denom = batch_size * context
        if target_tokens % denom:
            continue

        try:
            model.zero_grad(set_to_none=True)

            for gpu in (device_ids if use_multi and world >= 2 else [device.index or 0]):
                with torch.cuda.device(gpu):
                    torch.cuda.empty_cache()
                    torch.cuda.reset_peak_memory_stats(gpu)

            ids = torch.randint(
                0,
                model.cfg["vocab_size"],
                (batch_size, context),
                dtype=torch.long,
                device=device,
            )

            # ReplicaAutocast handles forward precision inside DP worker threads.
            # For single GPU, this outer context still controls the model forward.
            with amp(device, precision):
                loss = causal_loss(
                    model(ids),
                    ids,
                )

            loss.backward()

            peaks = {}
            enough_headroom = True
            check_devices = (
                device_ids
                if use_multi and world >= 2
                else [device.index or 0]
            )

            for gpu in check_devices:
                peak = torch.cuda.max_memory_allocated(gpu)
                total = torch.cuda.get_device_properties(gpu).total_memory

                # Optimizer states live with the master parameters on GPU0.
                reserve_gib = (
                    tc.get("optimizer_memory_reserve_gib", 2.5)
                    if gpu == check_devices[0]
                    else 0.5
                )
                reserve = int(float(reserve_gib) * (2**30))

                peaks[str(gpu)] = round(peak / 2**30, 2)
                if peak + reserve >= total:
                    enough_headroom = False

            print(
                {
                    "global_microbatch_probe": batch_size,
                    "per_gpu_examples": (
                        batch_size // world
                        if use_multi and world >= 2
                        else batch_size
                    ),
                    "peak_gib_by_gpu": peaks,
                    "enough_headroom": enough_headroom,
                }
            )

            del ids, loss
            model.zero_grad(set_to_none=True)

            for gpu in check_devices:
                with torch.cuda.device(gpu):
                    torch.cuda.empty_cache()

            if enough_headroom:
                chosen = batch_size
                break

        except torch.cuda.OutOfMemoryError:
            print(
                {
                    "global_microbatch_probe": batch_size,
                    "status": "OOM",
                }
            )
            model.zero_grad(set_to_none=True)
            for gpu in (
                device_ids
                if use_multi and world >= 2
                else [device.index or 0]
            ):
                with torch.cuda.device(gpu):
                    torch.cuda.empty_cache()

    model.train(original_training)

    if chosen is None:
        if use_multi and world >= 2:
            raise RuntimeError(
                "No two-GPU physical batch fit. Keep activation checkpointing off initially; "
                "if batch=2 truly OOMs, reduce model/context or deliberately enable a "
                "checkpointing implementation that passes backward tests."
            )
        raise RuntimeError("No microbatch candidate fit.")

    selected = dict(tc)
    selected["physical_batch_size"] = int(chosen)
    selected["gradient_accumulation_steps"] = (
        target_tokens // (chosen * context)
    )
    selected["eval_batch_size"] = max(
        world,
        min(8, max(chosen, chosen * 2)),
    )
    selected["auto_microbatch"] = False

    print(
        {
            "selected_global_physical_batch_size": selected["physical_batch_size"],
            "per_gpu_examples": (
                selected["physical_batch_size"] // world
                if use_multi and world >= 2
                else selected["physical_batch_size"]
            ),
            "gradient_accumulation_steps": selected["gradient_accumulation_steps"],
            "tokens_per_optimizer_step": target_tokens,
        }
    )
    return selected

def rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
    }


def restore_rng(rng):
    random.setstate(rng["python"])
    np.random.set_state(rng["numpy"])
    torch.set_rng_state(rng["torch"])
    if rng["cuda"] is not None:
        torch.cuda.set_rng_state_all(
            rng["cuda"]
        )


def identity(
    model,
    tc,
    dataset,
    tokenizer_path,
    precision,
):
    """Strict semantic resume identity; runtime is logged separately."""
    core_model = unwrap_model(model)
    return {
        "model": core_model.cfg,
        "training": tc,
        "precision": precision,
        "dataset": dataset.manifest["manifest_hash"],
        "tokenizer": data.tokenizer_identity(
            tokenizer_path
        )["sha256"],
        "implementation": {
            name: data.file_hash(
                Path(__file__).with_name(name)
            )
            for name in (
                "legal_manual_model.py",
                "legal_manual_data.py",
                "legal_manual_training.py",
            )
        },
    }


def save_checkpoint(
    root,
    model,
    optimizer,
    scheduler,
    scaler,
    state,
    ident,
    tc,
):
    target = root / (
        f"checkpoint-{state['step']:08d}.pt"
    )
    assert not target.exists(), (
        "Refusing to overwrite checkpoint"
    )

    temp = root / (
        target.name + ".partial"
    )

    core_model = unwrap_model(model)

    with temp.open("wb") as f:
        torch.save(
            {
                "model": core_model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "state": state,
                "identity": ident,
                "rng": rng_state(),
            },
            f,
        )
        f.flush()
        os.fsync(f.fileno())

    os.replace(temp, target)

    owned = sorted(
        root.glob("checkpoint-????????.pt")
    )
    for old in owned[
        : -tc["keep_checkpoints"]
    ]:
        assert (
            old.resolve().parent
            == root.resolve()
            and not old.is_symlink()
        )
        old.unlink()

    return target


def train(
    model,
    train_data,
    val_data,
    tc,
    output_dir,
    tokenizer_path,
    run_name,
    resume_from=None,
    stop_after_step=None,
):
    core_model = unwrap_model(model)
    mc = core_model.cfg
    p = plan(mc, tc)

    assert (
        train_data.context
        == val_data.context
        == mc["context_length"]
    )
    assert (
        train_data.manifest["manifest_hash"]
        == val_data.manifest["manifest_hash"]
    )
    assert (
        mc["vocab_size"]
        == train_data.manifest["vocab_size"]
    )
    assert (
        data.tokenizer_identity(
            tokenizer_path
        )["sha256"]
        == train_data.manifest["identity"][
            "tokenizer"
        ]["sha256"]
    )

    device = next(model.parameters()).device
    precision = precision_for(
        device,
        tc["precision"],
    )
    ident = identity(
        model,
        tc,
        train_data,
        tokenizer_path,
        precision,
    )

    root = Path(output_dir) / run_name
    assert (
        Path(run_name).name == run_name
        and run_name not in (".", "..")
    )
    root.mkdir(
        parents=True,
        exist_ok=True,
    )

    optimizer, scheduler, fused_optimizer = (
        optimizer_scheduler(
            model,
            mc,
            tc,
        )
    )
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=precision == "fp16",
    )

    state = {
        "step": 0,
        "tokens": 0,
        "cursor": 0,
        "elapsed": 0.0,
        "validation": {},
    }

    if resume_from:
        saved = torch.load(
            resume_from,
            map_location="cpu",
            weights_only=False,
        )
        assert (
            saved["identity"] == ident
        ), "Config/data/code changed; exact resume refused."

        core_model.load_state_dict(
            saved["model"]
        )
        optimizer.load_state_dict(
            saved["optimizer"]
        )
        scheduler.load_state_dict(
            saved["scheduler"]
        )
        scaler.load_state_dict(
            saved["scaler"]
        )
        state = saved["state"]
        restore_rng(saved["rng"])

        existing = list(
            root.glob("checkpoint-*.pt")
        )
        assert not any(
            int(x.stem.split("-")[-1])
            > state["step"]
            for x in existing
        ), "Output contains a newer checkpoint."
    else:
        assert (
            not list(
                root.glob("checkpoint-*")
            )
            and not (
                root / "training.jsonl"
            ).exists()
        ), "Use resume or a new run name."

        seed_all(
            tc["seed"],
            tc.get(
                "strict_determinism",
                False,
            ),
        )

    assert (
        state["tokens"]
        == state["step"]
        * p["tokens_per_step"]
    )
    assert (
        state["cursor"]
        == state["step"]
        * tc["gradient_accumulation_steps"]
    )

    target = (
        p["steps"]
        if stop_after_step is None
        else min(
            stop_after_step,
            p["steps"],
        )
    )
    assert target > state["step"]

    data.write_json(
        root / "model_config.json",
        mc,
    )
    data.write_json(
        root / "training_config.json",
        tc,
    )

    manifest = {
        "run_name": run_name,
        "identity": ident,
        "runtime": runtime(),
        "parameters": parameter_counts(core_model),
        "token_plan": p,
        "fused_optimizer": fused_optimizer,
        "special_tokens": {
            k: getattr(
                data.load_tokenizer(
                    tokenizer_path
                ),
                k,
            )
            for k in (
                "pad_token_id",
                "unk_token_id",
                "bos_token_id",
                "eos_token_id",
            )
        },
        "training_files": train_data.manifest[
            "identity"
        ]["sources"],
        "split_seed": train_data.manifest[
            "identity"
        ]["split_seed"],
    }
    data.write_json(
        root / "training_manifest.json",
        manifest,
    )

    sampler = data.ResumableBatches(
        len(train_data),
        tc["physical_batch_size"],
        state["cursor"],
        (
            target - state["step"]
        )
        * tc["gradient_accumulation_steps"],
        tc["seed"],
    )

    batches = iter(
        DataLoader(
            train_data,
            batch_sampler=sampler,
            **loader_options(tc),
        )
    )

    start = time.perf_counter()
    old_elapsed = state["elapsed"]
    start_tokens = state["tokens"]
    model.train()
    latest = None

    log_path = root / "training.jsonl"
    with log_path.open(
        "a",
        encoding="utf-8",
        buffering=1,
    ) as log:
        while state["step"] < target:
            optimizer.zero_grad(
                set_to_none=True
            )

            loss_sum = None
            lr = optimizer.param_groups[0][
                "lr"
            ]

            for _ in range(
                tc["gradient_accumulation_steps"]
            ):
                ids = next(batches)[
                    "input_ids"
                ].to(
                    device,
                    non_blocking=True,
                )

                with amp(device, precision):
                    loss = causal_loss(
                        model(ids),
                        ids,
                    )

                detached = loss.detach()
                loss_sum = (
                    detached
                    if loss_sum is None
                    else loss_sum + detached
                )

                scaler.scale(
                    loss
                    / tc[
                        "gradient_accumulation_steps"
                    ]
                ).backward()

            scaler.unscale_(optimizer)
            norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                tc["max_grad_norm"],
                error_if_nonfinite=True,
            )

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            state["step"] += 1
            state["cursor"] += tc[
                "gradient_accumulation_steps"
            ]
            state["tokens"] += p[
                "tokens_per_step"
            ]

            is_final = (
                state["step"] == target
            )
            should_eval = (
                state["step"]
                % tc["eval_steps"]
                == 0
                or is_final
            )
            should_log = (
                state["step"]
                % tc["logging_steps"]
                == 0
                or should_eval
                or state["step"]
                % tc["checkpoint_steps"]
                == 0
            )

            if should_eval:
                eval_budget = (
                    tc[
                        "validation_token_budget"
                    ]
                    if is_final
                    else tc[
                        "eval_token_budget"
                    ]
                )
                state["validation"] = (
                    validate(
                        model,
                        val_data,
                        eval_budget,
                        tc["eval_batch_size"],
                        tc,
                        device,
                        precision,
                    )
                )

            seconds = (
                time.perf_counter()
                - start
            )
            state["elapsed"] = (
                old_elapsed + seconds
            )

            if should_log:
                # Only now synchronize scalar loss/norm to CPU.
                loss_value = float(
                    (
                        loss_sum
                        / tc[
                            "gradient_accumulation_steps"
                        ]
                    ).item()
                )
                norm_value = float(
                    norm.item()
                    if torch.is_tensor(norm)
                    else norm
                )

                metrics = {
                    "step": state["step"],
                    "tokens": state["tokens"],
                    "loss": loss_value,
                    "learning_rate": lr,
                    "gradient_norm": norm_value,
                    "elapsed": state["elapsed"],
                    "tokens_per_second": (
                        (
                            state["tokens"]
                            - start_tokens
                        )
                        / max(
                            seconds,
                            1e-9,
                        )
                    ),
                    "gpu_peak_allocated_gib": (
                        torch.cuda.max_memory_allocated(
                            device
                        )
                        / 2**30
                        if device.type == "cuda"
                        else 0
                    ),
                }

                if should_eval:
                    metrics.update(
                        state["validation"]
                    )

                log.write(
                    json.dumps(
                        metrics,
                        allow_nan=False,
                    )
                    + "\n"
                )
                print(
                    metrics,
                    flush=True,
                )

            if (
                state["step"]
                % tc["checkpoint_steps"]
                == 0
                or is_final
            ):
                latest = save_checkpoint(
                    root,
                    model,
                    optimizer,
                    scheduler,
                    scaler,
                    state,
                    ident,
                    tc,
                )

                log.flush()
                os.fsync(log.fileno())

                row = {
                    "run_name": run_name,
                    "model_config": mc,
                    "training_config": tc,
                    "parameter_count": parameter_counts(
                        model
                    )["total"],
                    "tokens_processed": state[
                        "tokens"
                    ],
                    "checkpoint": str(
                        latest
                    ),
                    **state["validation"],
                }
                data.write_json(
                    root
                    / "ablation_result.json",
                    row,
                )
                data.write_json(
                    root / "progress.json",
                    state,
                )

    return {
        "model": model,
        "state": state,
        "checkpoint": str(latest),
    }


Writing legal_manual_training.py


In [17]:
import legal_manual_training as training

print("Requested precision:", training.precision_for(DEVICE, TRAIN_CONFIG["precision"]))
print("Initial plan:", training.plan(MODEL_CONFIG, TRAIN_CONFIG))


print("Configured multi-GPU:", TRAIN_CONFIG.get("use_multi_gpu"))
print("Configured device IDs:", TRAIN_CONFIG.get("device_ids"))
if TRAIN_CONFIG.get("require_two_gpus"):
    assert torch.cuda.device_count() >= 2, (
        "Select Kaggle accelerator T4 x2 and restart the session."
    )


Requested precision: bf16
Initial plan: {'tokens_per_microbatch': 8192, 'effective_batch_size': 32, 'tokens_per_step': 65536, 'steps': 3051, 'planned_train_tokens': 199950336, 'warmup': 62}


## Verify packed data and resolve resume state


In [18]:
train_data = data.PackedDataset(
    TOKEN_STORE,
    "train",
    MODEL_CONFIG["context_length"],
    verify_hashes=True,
)
val_data = data.PackedDataset(
    TOKEN_STORE,
    "validation",
    MODEL_CONFIG["context_length"],
    verify_hashes=True,
)

print("Train blocks:", len(train_data))
print("Validation blocks:", len(val_data))

run_dir = OUTPUT_DIR / RUN_NAME
existing = sorted(run_dir.glob("checkpoint-????????.pt")) if run_dir.exists() else []
RESUME_FROM = existing[-1] if (AUTO_RESUME and existing) else None

print("Resume checkpoint:", RESUME_FROM)

# If resuming, reuse the exact previously selected microbatch configuration.
saved_train_cfg = run_dir / "training_config.json"
if RESUME_FROM is not None and saved_train_cfg.exists():
    TRAIN_CONFIG = data.read_json(saved_train_cfg)
    print("Loaded saved training config for exact resume.")


Train blocks: 97701
Validation blocks: 2457
Resume checkpoint: /kaggle/working/legal-lm-phase1-optimized/baseline_71m_200m_optimized/checkpoint-00003051.pt
Loaded saved training config for exact resume.


## Training and resume


In [19]:
run_dir = OUTPUT_DIR / RUN_NAME

checkpoints = sorted(run_dir.glob("checkpoint-*.pt"))

if checkpoints:
    RESUME_FROM = checkpoints[-1]
    print("Resuming from:", RESUME_FROM)
else:
    RESUME_FROM = None
    print("No checkpoint found. Starting from scratch.")

Resuming from: /kaggle/working/legal-lm-phase1-optimized/baseline_71m_200m_optimized/checkpoint-00003051.pt


In [20]:
result = None
model = None
base_model = None

if RUN_FULL_TRAINING:
    assert DEVICE.type == "cuda", "Use a GPU for production training."

    if TRAIN_CONFIG.get("require_two_gpus", False):
        assert torch.cuda.device_count() >= 2, (
            "Two-GPU run requested. In Kaggle select Accelerator = T4 x2 "
            "and restart the notebook session."
        )

    # Exact resume must restore the training configuration BEFORE the
    # DataParallel/autocast wrapper is created.
    if RESUME_FROM is not None:
        saved_config_path = run_dir / "training_config.json"
        assert saved_config_path.is_file(), (
            "Resume checkpoint exists but training_config.json is missing."
        )
        TRAIN_CONFIG = json.loads(
            saved_config_path.read_text(encoding="utf-8")
        )

        if TRAIN_CONFIG.get("require_two_gpus", False):
            assert torch.cuda.device_count() >= 2

        print("Restored saved training configuration for exact resume.")

    training.seed_all(
        SEED,
        TRAIN_CONFIG.get("strict_determinism", False),
    )

    # Canonical model lives on GPU 0.
    base_model = LegalLanguageModel(MODEL_CONFIG).to(DEVICE)

    assert (
        50_000_000
        <= parameter_counts(base_model)["total"]
        <= 300_000_000
    )

    # Wrap BEFORE the fit probe so the probe exercises the same two-GPU path
    # that production training will use.
    model = training.prepare_training_model(
        base_model,
        TRAIN_CONFIG,
        DEVICE,
    )

    if RESUME_FROM is None:
        TRAIN_CONFIG = training.autotune_microbatch(
            model,
            TRAIN_CONFIG,
            DEVICE,
        )

    plan = training.plan(MODEL_CONFIG, TRAIN_CONFIG)
    required_blocks = (
        plan["steps"]
        * TRAIN_CONFIG["gradient_accumulation_steps"]
        * TRAIN_CONFIG["physical_batch_size"]
    )

    print("Selected training config:")
    print(json.dumps(TRAIN_CONFIG, indent=2))
    print("Training plan:", plan)
    print("Required train blocks:", required_blocks)
    print("Visible GPUs:", torch.cuda.device_count())

    if TRAIN_CONFIG.get("use_multi_gpu", False):
        device_ids = [
            int(i)
            for i in TRAIN_CONFIG["device_ids"]
            if int(i) < torch.cuda.device_count()
        ]
        print(
            "2-GPU DataParallel active:",
            {
                "device_ids": device_ids,
                "global_physical_batch": TRAIN_CONFIG["physical_batch_size"],
                "approx_examples_per_gpu": (
                    TRAIN_CONFIG["physical_batch_size"] // len(device_ids)
                ),
            },
        )

    assert len(train_data) >= required_blocks, (
        "Packed training data is smaller than the selected training plan."
    )

    result = training.train(
        model,
        train_data,
        val_data,
        TRAIN_CONFIG,
        OUTPUT_DIR,
        TOKENIZER_PATH,
        RUN_NAME,
        resume_from=RESUME_FROM,
        stop_after_step=SESSION_STOP_STEP,
    )
else:
    print("Training disabled.")


Training disabled.


In [21]:
from huggingface_hub import HfApi, create_repo

# HF_TOKEN is already read from Kaggle Secrets in the configuration cell.
assert HF_TOKEN, "Kaggle secret HF_TOKEN is required."
api = HfApi(token=HF_TOKEN)

MODEL_BACKUP_REPO = "I-230482/cs781-phase1-71m-checkpoints"

create_repo(
    repo_id=MODEL_BACKUP_REPO,
    repo_type="model",
    private=True,
    exist_ok=True,
    token=HF_TOKEN,
)


Repos ready.


In [22]:
run_dir = OUTPUT_DIR / RUN_NAME

api.upload_folder(
    folder_path=str(run_dir),
    repo_id=MODEL_BACKUP_REPO,
    repo_type="model",
    token=HF_TOKEN,
    commit_message="Backup training checkpoint",
)

print("Checkpoint backup complete.")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Checkpoint backup complete.


In [23]:
api.upload_folder(
    folder_path=str(TOKEN_STORE),
    repo_id=TOKEN_BACKUP_REPO,
    repo_type="dataset",
    token=HF_TOKEN,
    commit_message="Backup 200M Phase-1 token store",
)

print("Token-store backup complete.")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Token-store backup complete.


## Qualitative generation


In [24]:
PROMPT = "The court considered the submissions of both parties and held that"

if RUN_GENERATION:
    checkpoint_path = (
        Path(result["checkpoint"])
        if result is not None
        else RESUME_FROM
    )
    assert checkpoint_path is not None and checkpoint_path.is_file()

    saved = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )
    assert saved["identity"]["tokenizer"] == TOKENIZER_HASH

    inference_model = LegalLanguageModel(
        saved["identity"]["model"]
    ).to(DEVICE)
    inference_model.load_state_dict(saved["model"])
    inference_model.eval()

    text = continuation_words(
        inference_model,
        tokenizer,
        PROMPT,
        words=128,
        max_new_tokens=512,
        temperature=0.8,
        top_p=0.95,
        seed=SEED,
        precision="auto",
    )

    print("PROMPT\n", PROMPT)
    print("\n128-WORD CONTINUATION\n", text)


PROMPT
 The court considered the submissions of both parties and held that

128-WORD CONTINUATION
 the court below has held that where the facts of the case are of the considered opinion of the Court below as well as the law laid down by this Court in the case of Amrawati and another Vs. State of U.P. reported in 2004 (57) ALR 290 as well as judgement passed by Hon'ble Apex Court reported in 2009 (3) ADJ 322 (SC).

The application u/s 482 Cr.P.C. is allowed.

However, it is directed that in case the applicant surrenders before the court below within thirty days from today and apply for bail, his prayer for bail shall be considered and decided in view of the settled law laid by this Court in the case of Amrawati and another Vs. State of U.P. reported in 2004 (57)


## Predictions and submission


In [32]:
from pathlib import Path
from collections import defaultdict
import contextlib
import gc
import json
import re
import zipfile

import torch
from huggingface_hub import hf_hub_download


# ============================================================
# CONFIGURATION
# ============================================================

RUN_SUBMISSION = True

FINAL_CHECKPOINT = (
    OUTPUT_DIR
    / RUN_NAME
    / "checkpoint-00003051.pt"
)

TEST_FILENAME = "Phase_1_Pretraining/Test/test.json"

SUBMISSION_WORDS = 128

TEMPERATURE = 0.1
TOP_P = 0.95

# Safety limit only.
# Stall detection should normally stop a broken generation
# long before reaching this.
MAX_TOTAL_GENERATED_TOKENS = 4096

# Decode/check word count periodically.
CHECK_WORD_EVERY = 8

# If this many generated BPE tokens pass without increasing
# the whitespace-word count, treat generation as pathological.
STALL_TOKEN_LIMIT = 384

# Prevent exact token n-gram loops.
# First attempt uses 4-gram blocking.
# A retry uses stronger 3-gram blocking.
FIRST_NO_REPEAT_NGRAM = 4
RETRY_NO_REPEAT_NGRAM = 3

# Mild anti-repetition penalty on recently used token IDs.
REPETITION_PENALTY = 1.10
REPETITION_WINDOW = 256

# If context fills, rebuild using only the most recent 1536
# tokens, leaving ~512 positions for fast cached decoding.
ROLLING_REBUILD_LENGTH = 1536

# Number of automatic attempts for a pathological prompt.
MAX_GENERATION_ATTEMPTS = 3


# ============================================================
# OUTPUT PATHS
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROGRESS_PATH = (
    OUTPUT_DIR
    / "submission_progress.json"
)

PREDICTION_PATH = (
    OUTPUT_DIR
    / "prediction.json"
)

ZIP_PATH = (
    OUTPUT_DIR
    / "submission.zip"
)


# ============================================================
# TEST DATA PARSER
# ============================================================

def _test_records(obj):

    if isinstance(obj, dict):

        for key, value in obj.items():

            if isinstance(value, dict):
                text = value.get(
                    "text",
                    value.get("prompt"),
                )
            else:
                text = value

            if isinstance(text, str):
                yield str(key), text

        return

    if isinstance(obj, list):

        for i, value in enumerate(obj):

            assert isinstance(
                value,
                dict,
            )

            rid = value.get(
                "id",
                value.get("ID", i),
            )

            text = value.get(
                "text",
                value.get("prompt"),
            )

            assert isinstance(
                text,
                str,
            )

            yield str(rid), text

        return

    raise TypeError(
        "Unsupported test.json structure"
    )


# ============================================================
# AMP
# ============================================================

def _submission_amp(
    device,
    precision="auto",
):

    if device.type != "cuda":
        return contextlib.nullcontext()

    if precision == "auto":
        precision = (
            "bf16"
            if torch.cuda.is_bf16_supported()
            else "fp16"
        )

    if precision == "fp32":
        return contextlib.nullcontext()

    dtype = (
        torch.bfloat16
        if precision == "bf16"
        else torch.float16
    )

    return torch.autocast(
        device_type=device.type,
        dtype=dtype,
    )


# ============================================================
# WORD HELPERS
# ============================================================

def _word_spans(text):

    return list(
        re.finditer(
            r"\S+",
            text,
        )
    )


def _exact_first_words(
    text,
    words,
):

    spans = _word_spans(
        text
    )

    if len(spans) < words:
        return None

    result = (
        text[
            :spans[
                words - 1
            ].end()
        ]
        .strip()
    )

    assert (
        len(result.split())
        == words
    )

    return result


# ============================================================
# PATHOLOGICAL-GENERATION EXCEPTION
# ============================================================

class GenerationStalled(
    RuntimeError
):
    pass


# ============================================================
# REPETITION PENALTY
# ============================================================

def _apply_repetition_penalty(
    logits,
    generated,
    penalty,
    window,
):

    if (
        penalty <= 1.0
        or not generated
    ):
        return

    recent = generated[
        -window:
    ]

    unique_ids = list(
        set(recent)
    )

    if not unique_ids:
        return

    idx = torch.tensor(
        unique_ids,
        dtype=torch.long,
        device=logits.device,
    )

    values = logits[
        idx
    ]

    # Standard repetition-penalty form:
    #
    # positive logits -> divide
    # negative logits -> multiply
    #
    # This lowers probability of recently repeated token IDs.

    penalized = torch.where(
        values < 0,
        values * penalty,
        values / penalty,
    )

    logits[
        idx
    ] = penalized


# ============================================================
# NO-REPEAT NGRAM MAP
# ============================================================

def _update_ngram_map(
    generated,
    ngram_size,
    ngram_next,
):

    if (
        ngram_size <= 1
        or len(generated) < ngram_size
    ):
        return

    prefix = tuple(
        generated[
            -ngram_size:
            -1
        ]
    )

    next_token = (
        generated[-1]
    )

    ngram_next[
        prefix
    ].add(
        next_token
    )


def _blocked_ngram_tokens(
    generated,
    ngram_size,
    ngram_next,
):

    if (
        ngram_size <= 1
        or len(generated)
        < ngram_size - 1
    ):
        return set()

    prefix = tuple(
        generated[
            -(ngram_size - 1):
        ]
    )

    return ngram_next.get(
        prefix,
        set(),
    )


# ============================================================
# SAMPLE NEXT TOKEN
# ============================================================

def _sample_next_token(
    logits,
    tokenizer,
    generated,
    ngram_size,
    ngram_next,
    temperature,
    top_p,
    rng,
):

    next_logits = (
        logits[
            0,
            -1,
        ]
        .float()
        .clone()
    )

    # --------------------------------------------------------
    # Mild repetition penalty
    # --------------------------------------------------------

    _apply_repetition_penalty(
        next_logits,
        generated,
        REPETITION_PENALTY,
        REPETITION_WINDOW,
    )

    # --------------------------------------------------------
    # Never output special tokens.
    #
    # EOS is deliberately forbidden because submission requires
    # exactly 128 whitespace-delimited words.
    # --------------------------------------------------------

    for special in (
        tokenizer.pad_token_id,
        tokenizer.bos_token_id,
        tokenizer.unk_token_id,
        tokenizer.eos_token_id,
    ):

        if special is not None:

            next_logits[
                special
            ] = float(
                "-inf"
            )

    # Save version before ngram blocking.
    before_ngram = (
        next_logits.clone()
    )

    # --------------------------------------------------------
    # Block tokens that would recreate a previous n-gram.
    # This directly breaks loops such as:
    #
    # .P.P.P.P.P.P.P...
    # --------------------------------------------------------

    blocked = (
        _blocked_ngram_tokens(
            generated,
            ngram_size,
            ngram_next,
        )
    )

    for token_id in blocked:

        next_logits[
            token_id
        ] = float(
            "-inf"
        )

    # If ngram blocking somehow removed every possible token,
    # fall back to repetition-penalized logits rather than crash.

    if not torch.isfinite(
        next_logits
    ).any():

        next_logits = (
            before_ngram
        )

    if not torch.isfinite(
        next_logits
    ).any():

        raise RuntimeError(
            "All next-token logits "
            "are invalid."
        )

    # --------------------------------------------------------
    # Greedy
    # --------------------------------------------------------

    if temperature == 0:

        return int(
            next_logits.argmax().item()
        )

    # --------------------------------------------------------
    # Temperature + nucleus sampling
    # --------------------------------------------------------

    sorted_logits, order = (
        torch.sort(
            next_logits
            / temperature,
            descending=True,
        )
    )

    probs = (
        sorted_logits.softmax(-1)
    )

    cumulative = (
        probs.cumsum(-1)
    )

    remove = (
        cumulative
        - probs
    ) > top_p

    sorted_logits[
        remove
    ] = float(
        "-inf"
    )

    filtered_probs = (
        sorted_logits.softmax(-1)
    )

    sampled_position = (
        torch.multinomial(
            filtered_probs,
            num_samples=1,
            generator=rng,
        )
    )

    next_id = (
        order[
            sampled_position
        ]
        .item()
    )

    return int(
        next_id
    )


# ============================================================
# ONE GENERATION ATTEMPT
# ============================================================

@torch.inference_mode()
def _generate_exact_words_once(
    model,
    tokenizer,
    prompt,
    words,
    seed,
    ngram_size,
    temperature,
    top_p,
    precision="auto",
):

    prompt_ids = (
        tokenizer.encode(
            prompt,
            add_special_tokens=False,
            split_special_tokens=True,
        )
    )

    if not prompt_ids:

        raise ValueError(
            "Prompt tokenized to "
            "zero tokens."
        )

    context_length = int(
        model.cfg[
            "context_length"
        ]
    )

    if (
        len(prompt_ids)
        > context_length
    ):

        raise ValueError(
            f"Prompt has "
            f"{len(prompt_ids)} tokens "
            f"but model context is "
            f"{context_length}."
        )

    if not (
        0
        < ROLLING_REBUILD_LENGTH
        < context_length
    ):

        raise ValueError(
            "ROLLING_REBUILD_LENGTH "
            "must be between 1 and "
            "context_length - 1."
        )

    device = next(
        model.parameters()
    ).device

    rng = torch.Generator(
        device=device
    ).manual_seed(
        seed
    )

    was_training = (
        model.training
    )

    model.eval()

    generated = []

    all_tokens = list(
        prompt_ids
    )

    ngram_next = defaultdict(
        set
    )

    best_word_count = 0

    token_count_at_last_word_progress = 0

    rebuild_count = 0

    try:

        # ====================================================
        # INITIAL PROMPT PASS
        # ====================================================

        x = torch.tensor(
            [prompt_ids],
            dtype=torch.long,
            device=device,
        )

        with _submission_amp(
            device,
            precision,
        ):

            logits, cache = (
                model(
                    x,
                    use_cache=True,
                )
            )

        # ====================================================
        # TOKEN GENERATION
        # ====================================================

        for step in range(
            MAX_TOTAL_GENERATED_TOKENS
        ):

            next_id = (
                _sample_next_token(
                    logits,
                    tokenizer,
                    generated,
                    ngram_size,
                    ngram_next,
                    temperature,
                    top_p,
                    rng,
                )
            )

            generated.append(
                next_id
            )

            all_tokens.append(
                next_id
            )

            _update_ngram_map(
                generated,
                ngram_size,
                ngram_next,
            )

            # ------------------------------------------------
            # Check decoded whitespace-word count
            # ------------------------------------------------

            should_check = (
                len(generated)
                % CHECK_WORD_EVERY
                == 0
                or
                step + 1
                == MAX_TOTAL_GENERATED_TOKENS
            )

            if should_check:

                text = (
                    tokenizer.decode(
                        generated,
                        skip_special_tokens=True,
                        clean_up_tokenization_spaces=False,
                    )
                )

                spans = (
                    _word_spans(
                        text
                    )
                )

                current_words = len(
                    spans
                )

                # --------------------------------------------
                # Word-count progress
                # --------------------------------------------

                if (
                    current_words
                    > best_word_count
                ):

                    best_word_count = (
                        current_words
                    )

                    token_count_at_last_word_progress = (
                        len(
                            generated
                        )
                    )

                # --------------------------------------------
                # Success
                # --------------------------------------------

                if (
                    current_words
                    >= words
                ):

                    result = (
                        text[
                            :spans[
                                words - 1
                            ].end()
                        ]
                        .strip()
                    )

                    assert (
                        len(
                            result.split()
                        )
                        == words
                    )

                    return result

                # --------------------------------------------
                # Detect pathological no-whitespace repetition.
                #
                # Example:
                # .P.P.P.P.P.P.P...
                # --------------------------------------------

                stagnant_tokens = (
                    len(generated)
                    - token_count_at_last_word_progress
                )

                if (
                    stagnant_tokens
                    >= STALL_TOKEN_LIMIT
                ):

                    raise GenerationStalled(
                        f"Word count stalled at "
                        f"{current_words} words "
                        f"for {stagnant_tokens} "
                        f"generated tokens. "
                        f"Last fragment: "
                        f"{text[-300:]!r}"
                    )

            # =================================================
            # GET LOGITS FOR NEXT TOKEN
            # =================================================

            current_cache_length = (
                int(
                    cache[
                        0
                    ][
                        0
                    ].shape[
                        -2
                    ]
                )
            )

            # -------------------------------------------------
            # FAST KV-CACHE PATH
            # -------------------------------------------------

            if (
                current_cache_length
                < context_length
            ):

                x = torch.tensor(
                    [[next_id]],
                    dtype=torch.long,
                    device=device,
                )

                with _submission_amp(
                    device,
                    precision,
                ):

                    logits, cache = (
                        model(
                            x,
                            past_key_values=cache,
                            use_cache=True,
                        )
                    )

            # -------------------------------------------------
            # CONTEXT FULL:
            # rebuild only newest 1536 tokens.
            #
            # This prevents the expensive mistake of rebuilding
            # all 2048 tokens after EVERY generated token.
            # -------------------------------------------------

            else:

                rebuild_count += 1

                window = (
                    all_tokens[
                        -ROLLING_REBUILD_LENGTH:
                    ]
                )

                x = torch.tensor(
                    [window],
                    dtype=torch.long,
                    device=device,
                )

                with _submission_amp(
                    device,
                    precision,
                ):

                    logits, cache = (
                        model(
                            x,
                            use_cache=True,
                        )
                    )

        # ====================================================
        # SAFETY LIMIT
        # ====================================================

        final_text = (
            tokenizer.decode(
                generated,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )
        )

        final_words = len(
            _word_spans(
                final_text
            )
        )

        raise GenerationStalled(
            f"Reached "
            f"{MAX_TOTAL_GENERATED_TOKENS} "
            f"generated tokens with only "
            f"{final_words} words. "
            f"Rebuilds={rebuild_count}. "
            f"Last fragment="
            f"{final_text[-300:]!r}"
        )

    finally:

        model.train(
            was_training
        )


# ============================================================
# AUTOMATIC RETRY WRAPPER
# ============================================================

def generate_exact_words(
    model,
    tokenizer,
    prompt,
    words=128,
    seed=42,
    temperature=0.1,
    top_p=0.95,
    precision="auto",
):

    errors = []

    for attempt in range(
        MAX_GENERATION_ATTEMPTS
    ):

        # First attempt:
        #   4-token no-repeat constraint
        #
        # Retries:
        #   stronger 3-token constraint

        ngram_size = (
            FIRST_NO_REPEAT_NGRAM
            if attempt == 0
            else RETRY_NO_REPEAT_NGRAM
        )

        attempt_seed = (
            seed
            + attempt
            * 100_003
        )

        print(
            f"    generation attempt "
            f"{attempt + 1}/"
            f"{MAX_GENERATION_ATTEMPTS} "
            f"| no_repeat_ngram="
            f"{ngram_size}",
            flush=True,
        )

        try:

            return (
                _generate_exact_words_once(
                    model=model,
                    tokenizer=tokenizer,
                    prompt=prompt,
                    words=words,
                    seed=attempt_seed,
                    ngram_size=ngram_size,
                    temperature=temperature,
                    top_p=top_p,
                    precision=precision,
                )
            )

        except GenerationStalled as e:

            errors.append(
                str(e)
            )

            print(
                "    pathological "
                "generation detected; "
                "retrying.",
                flush=True,
            )

            print(
                "    ",
                str(e)[:500],
                flush=True,
            )

    raise RuntimeError(
        "All generation attempts "
        "failed.\n"
        + "\n".join(
            f"Attempt {i + 1}: {msg}"
            for i, msg
            in enumerate(errors)
        )
    )


# ============================================================
# LOAD SAVED PROGRESS
# ============================================================

def load_progress():

    if not (
        PROGRESS_PATH.is_file()
    ):

        return {}

    try:

        obj = json.loads(
            PROGRESS_PATH.read_text(
                encoding="utf-8"
            )
        )

    except Exception:

        return {}

    if not isinstance(
        obj,
        dict,
    ):

        return {}

    good = {}

    for rid, value in (
        obj.items()
    ):

        if not isinstance(
            value,
            dict,
        ):
            continue

        text = value.get(
            "text"
        )

        if not isinstance(
            text,
            str,
        ):
            continue

        if (
            len(text.split())
            != SUBMISSION_WORDS
        ):
            continue

        good[
            str(rid)
        ] = {
            "text": text
        }

    return good


# ============================================================
# ATOMIC PROGRESS SAVE
# ============================================================

def save_progress(
    predictions,
):

    temp = (
        OUTPUT_DIR
        / "submission_progress.tmp"
    )

    temp.write_text(
        json.dumps(
            predictions,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    temp.replace(
        PROGRESS_PATH
    )


# ============================================================
# MAIN SUBMISSION GENERATION
# ============================================================

if RUN_SUBMISSION:

    checkpoint_path = Path(
        FINAL_CHECKPOINT
    )

    assert (
        checkpoint_path.is_file()
    ), (
        f"Checkpoint not found: "
        f"{checkpoint_path}"
    )

    print(
        "Using checkpoint:",
        checkpoint_path,
    )

    # ========================================================
    # TEST FILE
    # ========================================================

    test_path = Path(
        hf_hub_download(
            repo_id=HF_DATASET_REPO,
            repo_type="dataset",
            filename=TEST_FILENAME,
            revision=DATASET_REVISION,
            token=HF_TOKEN,
        )
    )

    print(
        "Test file:",
        test_path,
    )

    test_obj = json.loads(
        test_path.read_text(
            encoding="utf-8"
        )
    )

    records = list(
        _test_records(
            test_obj
        )
    )

    assert records

    expected_ids = [
        rid
        for rid, _
        in records
    ]

    assert (
        len(expected_ids)
        == len(
            set(expected_ids)
        )
    ), (
        "Duplicate test IDs."
    )

    expected_id_set = set(
        expected_ids
    )

    print(
        "Test prompts:",
        len(records),
    )

    # ========================================================
    # EXISTING PROGRESS
    # ========================================================

    predictions = (
        load_progress()
    )

    predictions = {
        rid: value
        for rid, value
        in predictions.items()
        if rid
        in expected_id_set
    }

    print(
        "Already completed:",
        len(predictions),
        "/",
        len(records),
    )

    # ========================================================
    # CLEAR PREVIOUS GPU MODEL
    # ========================================================

    if (
        "submit_model"
        in globals()
    ):

        try:
            del submit_model
        except Exception:
            pass

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

    # ========================================================
    # LOAD CHECKPOINT
    # ========================================================

    print(
        "Loading model..."
    )

    saved = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    assert (
        saved[
            "identity"
        ][
            "tokenizer"
        ]
        == TOKENIZER_HASH
    ), (
        "Tokenizer/checkpoint mismatch."
    )

    submit_model = (
        LegalLanguageModel(
            saved[
                "identity"
            ][
                "model"
            ]
        )
        .to(DEVICE)
    )

    submit_model.load_state_dict(
        saved["model"]
    )

    submit_model.eval()

    del saved

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(
        "Model loaded."
    )

    # ========================================================
    # GENERATE
    # ========================================================

    failed_ids = []

    for n, (
        rid,
        prompt,
    ) in enumerate(
        records,
        start=1,
    ):

        # ----------------------------------------------------
        # Already safely generated
        # ----------------------------------------------------

        if rid in predictions:

            print(
                f"[{n}/{len(records)}] "
                f"id={rid} "
                f"SKIPPED "
                f"(already saved)",
                flush=True,
            )

            continue

        prompt_tokens = len(
            tokenizer.encode(
                prompt,
                add_special_tokens=False,
                split_special_tokens=True,
            )
        )

        print()
        print(
            f"[{n}/{len(records)}] "
            f"id={rid} | "
            f"prompt_tokens="
            f"{prompt_tokens}",
            flush=True,
        )

        try:

            continuation = (
                generate_exact_words(
                    model=submit_model,
                    tokenizer=tokenizer,
                    prompt=prompt,
                    words=(
                        SUBMISSION_WORDS
                    ),
                    seed=SEED + n,
                    temperature=(
                        TEMPERATURE
                    ),
                    top_p=TOP_P,
                    precision="auto",
                )
            )

        except Exception as e:

            print(
                f"    FAILED id={rid}: "
                f"{type(e).__name__}: "
                f"{e}",
                flush=True,
            )

            failed_ids.append(
                rid
            )

            # Do not let one prompt block every later prompt.
            continue

        actual_words = len(
            continuation.split()
        )

        assert (
            actual_words
            == SUBMISSION_WORDS
        ), (
            f"id={rid}: "
            f"{actual_words} words"
        )

        predictions[
            rid
        ] = {
            "text": continuation
        }

        # Save IMMEDIATELY after every successful ID.
        save_progress(
            predictions
        )

        print(
            f"    DONE | "
            f"{actual_words} words | "
            f"saved",
            flush=True,
        )

    # ========================================================
    # REPORT ANY REMAINING FAILURES
    # ========================================================

    missing_ids = [
        rid
        for rid
        in expected_ids
        if rid
        not in predictions
    ]

    if missing_ids:

        print()
        print(
            "=" * 60
        )

        print(
            "NOT READY TO ZIP"
        )

        print(
            "=" * 60
        )

        print(
            "Missing IDs:",
            missing_ids,
        )

        print(
            "Successful predictions:",
            len(predictions),
            "/",
            len(records),
        )

        print(
            "Progress is safely stored at:",
            PROGRESS_PATH,
        )

    else:

        # ====================================================
        # FINAL VALIDATION
        # ====================================================

        assert (
            set(
                predictions.keys()
            )
            == expected_id_set
        )

        for rid, value in (
            predictions.items()
        ):

            assert isinstance(
                value,
                dict,
            )

            assert isinstance(
                value.get("text"),
                str,
            )

            count = len(
                value[
                    "text"
                ].split()
            )

            assert (
                count
                == SUBMISSION_WORDS
            ), (
                f"id={rid}: "
                f"{count} words"
            )

        # Preserve original test ordering.
        ordered_predictions = {
            rid: predictions[
                rid
            ]
            for rid, _
            in records
        }

        # ====================================================
        # WRITE prediction.json
        # ====================================================

        PREDICTION_PATH.write_text(
            json.dumps(
                ordered_predictions,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

        # ====================================================
        # ZIP
        # ====================================================

        with zipfile.ZipFile(
            ZIP_PATH,
            "w",
            compression=(
                zipfile.ZIP_DEFLATED
            ),
        ) as zf:

            zf.write(
                PREDICTION_PATH,
                arcname=(
                    "prediction.json"
                ),
            )

        # ====================================================
        # VERIFY ZIP CONTENT
        # ====================================================

        with zipfile.ZipFile(
            ZIP_PATH,
            "r",
        ) as zf:

            assert (
                zf.namelist()
                == [
                    "prediction.json"
                ]
            )

            zipped_obj = (
                json.loads(
                    zf.read(
                        "prediction.json"
                    ).decode(
                        "utf-8"
                    )
                )
            )

            assert (
                set(
                    zipped_obj.keys()
                )
                == expected_id_set
            )

            for rid, value in (
                zipped_obj.items()
            ):

                assert (
                    len(
                        value[
                            "text"
                        ].split()
                    )
                    == SUBMISSION_WORDS
                )

        # ====================================================
        # DONE
        # ====================================================

        print()
        print(
            "=" * 60
        )

        print(
            "SUBMISSION COMPLETE"
        )

        print(
            "=" * 60
        )

        print(
            "prediction.json:",
            PREDICTION_PATH,
        )

        print(
            "submission.zip:",
            ZIP_PATH,
        )

        print(
            "Predictions:",
            len(
                ordered_predictions
            ),
        )

        print(
            "Words each:",
            SUBMISSION_WORDS,
        )

        print(
            "ZIP contents:",
            [
                "prediction.json"
            ],
        )

Using checkpoint: /kaggle/working/legal-lm-phase1-optimized/baseline_71m_200m_optimized/checkpoint-00003051.pt
Test file: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json
Test prompts: 1000
Already completed: 50 / 1000
Loading model...
Model loaded.
[1/1000] id=1 SKIPPED (already saved)
[2/1000] id=2 SKIPPED (already saved)
[3/1000] id=3 SKIPPED (already saved)
[4/1000] id=4 SKIPPED (already saved)
[5/1000] id=5 SKIPPED (already saved)
[6/1000] id=6 SKIPPED (already saved)
[7/1000] id=7 SKIPPED (already saved)
[8/1000] id=8 SKIPPED (already saved)
[9/1000] id=9 SKIPPED (already saved)
[10/1000] id=10 SKIPPED (already saved)
[11/1000] id=11 SKIPPED (already saved)
[12/1000] id=12 SKIPPED (already saved)
[13/1000] id=13 SKIPPED (already saved)
[14/1000] id=14 SKIPPED (already saved)
[15/1000] id=15 SKIPPED (already saved)
[16/1000] id=16 SKIPPED (already saved)
[17/1000] id=17 SKI

## Cleanup / output inventory


In [33]:
if not RUN_FULL_TRAINING:
    train_data.close()
    val_data.close()

assert data.tokenizer_identity(TOKENIZER_PATH)["sha256"] == TOKENIZER_HASH

print("\nOutput files:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(
            path.relative_to(OUTPUT_DIR),
            f"{path.stat().st_size/2**20:.2f} MiB",
        )



Output files:
baseline_71m_200m_optimized/.cache/huggingface/.gitignore 0.00 MiB
baseline_71m_200m_optimized/.cache/huggingface/download/.gitattributes.lock 0.00 MiB
baseline_71m_200m_optimized/.cache/huggingface/download/.gitattributes.metadata 0.00 MiB
baseline_71m_200m_optimized/.cache/huggingface/download/ablation_result.json.lock 0.00 MiB
baseline_71m_200m_optimized/.cache/huggingface/download/ablation_result.json.metadata 0.00 MiB
baseline_71m_200m_optimized/.cache/huggingface/download/checkpoint-00001500.pt.lock 0.00 MiB
baseline_71m_200m_optimized/.cache/huggingface/download/checkpoint-00001500.pt.metadata 0.00 MiB
baseline_71m_200m_optimized/.cache/huggingface/download/checkpoint-00001750.pt.lock 0.00 MiB
baseline_71m_200m_optimized/.cache/huggingface/download/checkpoint-00001750.pt.metadata 0.00 MiB
baseline_71m_200m_optimized/.cache/huggingface/download/checkpoint-00003000.pt.lock 0.00 MiB
baseline_71m_200m_optimized/.cache/huggingface/download/checkpoint-00003000.pt.metada